# PyEO Forest Alerts: How to detect land cover changes between a time-series of Sentinel-2 images and a baseline image composite from the past using a trained machine learning classification model.

This notebook was adapted for PyEO on Windows for the Amazon BR-163 pilot.

## Safe execution modes

The companion `amazon_br163_working.ini` is delivered in **validation-safe mode** with all processing-stage flags disabled. In that state, restarting the kernel and running the notebook verifies imports, configuration, paths, existing outputs, and read-only QA without downloading imagery, rebuilding rasters, rewriting vector outputs, or deleting intermediates.

To run a processing phase deliberately, enable only the required flag in the INI, save the INI, restart the kernel, and run the notebook from the top. Restore the flag to `False` after the phase completes.

The notebook never relies on hard-coded recovery variables. Operational paths are read from the INI during each fresh kernel session.


## Downloading and pre-processing a time series of Sentinel-2 images for change detection

- This section will take us stepwise through the imagery query, download and cloud-masking aspects of the `run_acd_national.py` script, which runs the full PyEO pipeline from the command line in a terminal.  
- Jupyter notebooks provide a useful and engaging interface to understand the components of this script, so we will follow an extracted version throughout this notebook.

This section comprises several stages:   
1. Directory and Variable setup.
1. Querying for Sentinel-2 imagery that meets our search criteria.
1. Downloading the Sentinel-2 imagery identified from the Query.
1. If necessary, preprocess any L1C to L2A by applying atmospheric corrections. 
1. Cloud-masking the L2A imagery.
1. Classification of all Change Detection Images
1. Creation of Forest Alerts

## Setup: Requirements to use this Notebook

In [1]:
pwd

'C:\\GIS\\projects\\Amazon_BR163_PyEO\\03_notebooks'

In [2]:
%cd C:\GIS\src\pyeo

C:\GIS\src\pyeo


We did this in the previous notebook step-by-step. Here, we initialie the notebook in one code cell to speed up the process.

In [3]:
# ============================================================
# AMAZON BR163 MONITORING AND CHANGE-DETECTION INITIALISATION
# ============================================================

import shutil
import sys
import configparser
import argparse
import json
import os
import glob
import warnings
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal

from pyeo import (
    classification,
    filesystem_utilities,
    queries_and_downloads,
    raster_manipulation,
    vectorisation,
)

from pyeo.acd_national import (
    acd_initialisation,
    acd_config_to_log,
    acd_roi_tile_intersection,
)


# ------------------------------------------------------------
# 1. GDAL and warning behaviour
# ------------------------------------------------------------

gdal.UseExceptions()

warnings.simplefilter(
    "ignore",
    category=UserWarning,
)

print("Libraries successfully imported")


# ------------------------------------------------------------
# 2. Fixed Amazon project paths
# ------------------------------------------------------------

project_dir = Path(
    r"C:\GIS\projects\Amazon_BR163_PyEO"
)

pyeo_dir = Path(
    r"C:\GIS\src\pyeo"
)

config_path = (
    project_dir
    / "00_admin"
    / "amazon_br163_working.ini"
)

credentials_path_expected = Path(
    r"C:\GIS\secrets\pyeo_cdse.ini"
)

baseline_classified_raster = (
    project_dir
    / "06_outputs"
    / "composite_T21MYM_20240912T140049_full_class.tif"
)

amazon_model_path = (
    project_dir
    / "05_model"
    / "amazon_forest_clearing_extratrees.pkl"
)

monitoring_root = Path(
    r"C:\GIS\data\Amazon_BR163_PyEO"
    r"\tile_data\21MYM\monitoring"
)

monitoring_root.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 3. Required path checks
# ------------------------------------------------------------

required_paths = [
    pyeo_dir,
    config_path,
    credentials_path_expected,
    baseline_classified_raster,
    amazon_model_path,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {required_path}"
        )

print("Required Amazon project paths: PASS")


# ------------------------------------------------------------
# 4. Initialise PyEO using the working Amazon configuration
# ------------------------------------------------------------

config_dict, acd_log = acd_initialisation(
    str(config_path)
)

os.chdir(
    str(pyeo_dir)
)

acd_config_to_log(
    config_dict,
    acd_log,
)

print("Configuration loaded:", config_path)
print("PyEO directory:", pyeo_dir)


# ------------------------------------------------------------
# 5. Confirm critical configuration values
# ------------------------------------------------------------

configured_pyeo_dir = Path(
    config_dict["pyeo_dir"]
)

configured_credentials_path = Path(
    config_dict["credentials_path"]
)

configured_model_path = Path(
    config_dict["model_path"]
)

if configured_pyeo_dir.resolve() != pyeo_dir.resolve():
    raise RuntimeError(
        "The configured pyeo_dir does not match the "
        "validated PyEO repository.\n"
        f"Configured: {configured_pyeo_dir}\n"
        f"Expected: {pyeo_dir}"
    )

if (
    configured_credentials_path.resolve()
    != credentials_path_expected.resolve()
):
    raise RuntimeError(
        "The configuration credentials path does not match "
        "the validated CDSE credentials file.\n"
        f"Configured: {configured_credentials_path}\n"
        f"Expected: {credentials_path_expected}"
    )

if configured_model_path.resolve() != amazon_model_path.resolve():
    raise RuntimeError(
        "The configuration model path does not match the "
        "validated Amazon model.\n"
        f"Configured: {configured_model_path}\n"
        f"Expected: {amazon_model_path}"
    )

print("Critical configuration alignment: PASS")


# ------------------------------------------------------------
# 6. Build or read the ROI-to-tile intersection
# ------------------------------------------------------------

tilelist_filepath = acd_roi_tile_intersection(
    config_dict,
    acd_log,
)

if not os.path.isfile(tilelist_filepath):
    raise FileNotFoundError(
        f"Tile list was not created: {tilelist_filepath}"
    )

tile_table = pd.read_csv(
    tilelist_filepath
)

if "tile" not in tile_table.columns:
    raise KeyError(
        "The tile-intersection table does not contain "
        "the required 'tile' column"
    )

tile_values = (
    tile_table["tile"]
    .astype(str)
    .str.strip()
    .tolist()
)

if "21MYM" not in tile_values:
    raise RuntimeError(
        "Tile 21MYM was not found in the ROI intersection.\n"
        f"Tiles found: {tile_values}"
    )

tile_to_process = "21MYM"

print("Tile-intersection output:", tilelist_filepath)
print("Selected pilot tile:", tile_to_process)


# ------------------------------------------------------------
# 7. Resolve the validated tile directory
# ------------------------------------------------------------

individual_tile_directory_path = os.path.join(
    config_dict["tile_dir"],
    tile_to_process,
)

filesystem_utilities.create_folder_structure_for_tiles(
    individual_tile_directory_path
)

print(
    "Tile directory path:",
    individual_tile_directory_path,
)

print(
    "Folder structure build successfully finished"
)


# ------------------------------------------------------------
# 8. Initialise tile-specific logging
# ------------------------------------------------------------

tile_log_path = os.path.join(
    individual_tile_directory_path,
    "log",
    tile_to_process + ".log",
)

tile_log = filesystem_utilities.init_log_acd(
    log_path=tile_log_path,
    logger_name=f"pyeo_{tile_to_process}_monitoring",
)

tile_log.info(
    "Amazon BR163 monitoring notebook initialised"
)

tile_log.info(
    "Selected pilot tile: %s",
    tile_to_process,
)


# ------------------------------------------------------------
# 9. Read processing parameters from the configuration
# ------------------------------------------------------------

start_date = config_dict["start_date"]
end_date = config_dict["end_date"]

composite_start_date = config_dict["composite_start"]
composite_end_date = config_dict["composite_end"]

cloud_cover = config_dict["cloud_cover"]

cloud_certainty_threshold = (
    config_dict["cloud_certainty_threshold"]
)

model_path = str(
    amazon_model_path
)

sen2cor_path = config_dict["sen2cor_path"]
epsg = config_dict["epsg"]
bands = config_dict["bands"]
resolution = config_dict["resolution_string"]
out_resolution = config_dict["output_resolution"]

buffer_size = (
    config_dict["buffer_size_cloud_masking"]
)

buffer_size_composite = (
    config_dict[
        "buffer_size_cloud_masking_composite"
    ]
)

max_image_number = config_dict["download_limit"]

faulty_granule_threshold = (
    config_dict["faulty_granule_threshold"]
)

download_limit = config_dict["download_limit"]

skip_existing = config_dict["do_skip_existing"]
sieve = config_dict["sieve"]
from_classes = config_dict["from_classes"]
to_classes = config_dict["to_classes"]

download_source = config_dict["download_source"]


# ------------------------------------------------------------
# 10. Validate model and change classes
# ------------------------------------------------------------

if Path(model_path).resolve() != amazon_model_path.resolve():
    raise RuntimeError(
        "The active model is not the validated Amazon model"
    )

print("Active model:", model_path)
print("Configured from_classes:", from_classes)
print("Configured to_classes:", to_classes)

# Current Amazon model:
# 1 = Forest
# 2 = Clearing
#
# The first controlled transition must therefore be 1 -> 2.

expected_from_classes = [1]
expected_to_classes = [2]

if list(from_classes) != expected_from_classes:
    raise RuntimeError(
        "The configuration from_classes value is not aligned "
        "with the Amazon binary model.\n"
        f"Found: {from_classes}\n"
        f"Expected: {expected_from_classes}"
    )

if list(to_classes) != expected_to_classes:
    raise RuntimeError(
        "The configuration to_classes value is not aligned "
        "with the Amazon binary model.\n"
        f"Found: {to_classes}\n"
        f"Expected: {expected_to_classes}"
    )

print("Forest-to-Clearing transition alignment: PASS")


# ------------------------------------------------------------
# 11. Validate download source
# ------------------------------------------------------------

if download_source == "dataspace":
    tile_log.info(
        "Copernicus Data Space Ecosystem is the "
        "download source"
    )

elif download_source == "scihub":
    raise RuntimeError(
        "The configuration still uses the retired SciHub "
        "download source. Set download_source=dataspace."
    )

else:
    raise RuntimeError(
        f"Unsupported download source: {download_source}"
    )

tile_log.info(
    "Faulty Granule Threshold is set to: %s",
    config_dict["faulty_granule_threshold"],
)

tile_log.info(
    "Files below this threshold will not be downloaded"
)


# ------------------------------------------------------------
# 12. Validate and read credentials
# ------------------------------------------------------------

credentials_path = Path(
    config_dict["credentials_path"]
)

print("Credentials path:", credentials_path)

if not credentials_path.is_file():
    tile_log.error(
        "The credentials path does not exist: %s",
        credentials_path,
    )

    tile_log.error(
        "Current working directory: %s",
        os.getcwd(),
    )

    raise FileNotFoundError(
        f"Credentials file not found: {credentials_path}"
    )

conf = configparser.ConfigParser(
    allow_no_value=True,
    interpolation=None,
)

files_read = conf.read(
    credentials_path
)

if not files_read:
    raise RuntimeError(
        f"Credentials file could not be read: "
        f"{credentials_path}"
    )

if "dataspace" not in conf:
    raise KeyError(
        "The credentials file does not contain a "
        "[dataspace] section"
    )

required_credential_keys = [
    "user",
    "pass",
]

missing_credential_keys = [
    key
    for key in required_credential_keys
    if key not in conf["dataspace"]
]

if missing_credential_keys:
    raise KeyError(
        "Missing Data Space credential keys: "
        f"{missing_credential_keys}"
    )

credentials_dict = {
    "sent_2": {
        "user": conf["dataspace"]["user"],
        "pass": conf["dataspace"]["pass"],
    }
}

sen_user = credentials_dict["sent_2"]["user"]
sen_pass = credentials_dict["sent_2"]["pass"]

tile_log.info(
    "Successfully read processing arguments "
    "and credentials"
)

print("Credentials configuration: PASS")


# ------------------------------------------------------------
# 13. Resolve standard PyEO tile directories
# ------------------------------------------------------------

tile_log.info(
    "Creating tile directory paths if they do not exist"
)

change_image_dir = os.path.join(
    individual_tile_directory_path,
    "images",
)

l1_image_dir = os.path.join(
    individual_tile_directory_path,
    "images",
    "L1C",
)

l2_image_dir = os.path.join(
    individual_tile_directory_path,
    "images",
    "L2A",
)

l2_masked_image_dir = os.path.join(
    individual_tile_directory_path,
    "images",
    "cloud_masked",
)

categorised_image_dir = os.path.join(
    individual_tile_directory_path,
    "output",
    "classifications",
)

probability_image_dir = os.path.join(
    individual_tile_directory_path,
    "output",
    "probabilities",
)

sieved_image_dir = os.path.join(
    individual_tile_directory_path,
    "output",
    "sieved",
)

composite_dir = os.path.join(
    individual_tile_directory_path,
    "composite",
)

composite_l1_image_dir = os.path.join(
    individual_tile_directory_path,
    "composite",
    "L1C",
)

composite_l2_image_dir = os.path.join(
    individual_tile_directory_path,
    "composite",
    "L2A",
)

composite_l2_masked_image_dir = os.path.join(
    individual_tile_directory_path,
    "composite",
    "cloud_masked",
)

quicklook_dir = os.path.join(
    individual_tile_directory_path,
    "output",
    "quicklooks",
)


# ------------------------------------------------------------
# 14. Create dedicated monitoring directories
# ------------------------------------------------------------

monitoring_images_dir = (
    monitoring_root
    / "images"
)

monitoring_l1c_dir = (
    monitoring_images_dir
    / "L1C"
)

monitoring_l2a_dir = (
    monitoring_images_dir
    / "L2A"
)

monitoring_cloud_masked_dir = (
    monitoring_images_dir
    / "cloud_masked"
)

monitoring_composite_dir = (
    monitoring_root
    / "composite"
)

monitoring_output_dir = (
    monitoring_root
    / "output"
)

monitoring_log_dir = (
    monitoring_root
    / "log"
)

for directory_path in [
    monitoring_images_dir,
    monitoring_l1c_dir,
    monitoring_l2a_dir,
    monitoring_cloud_masked_dir,
    monitoring_composite_dir,
    monitoring_output_dir,
    monitoring_log_dir,
]:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

tile_log.info(
    "Successfully created standard and monitoring "
    "directory paths"
)

print("Monitoring directory structure: PASS")


# ------------------------------------------------------------
# 15. Final initialisation summary
# ------------------------------------------------------------

print()
print("=" * 68)
print("AMAZON MONITORING NOTEBOOK INITIALISATION: PASS")
print("=" * 68)
print("Pilot tile:", tile_to_process)
print("Configuration:", config_path)
print("Download source:", download_source)
print("Monitoring start:", start_date)
print("Monitoring end:", end_date)
print("Composite start:", composite_start_date)
print("Composite end:", composite_end_date)
print("CRS:", epsg)
print("Bands:", bands)
print("Output resolution:", out_resolution)
print("Model:", model_path)
print("Transition: Forest class 1 -> Clearing class 2")
print("Monitoring root:", monitoring_root)
print("=" * 68)

2026-07-30 02:41:45,668: WARNING: Windows or iOS detected; Patching GetVirtualMemArray. Some functions may not respond as expected.
C:\Users\moses\miniconda3\envs\pyeo_env\lib\abc.py:106: SHDeprecationWarning: AWS functionality will remain in the codebase for now, but won't be actively maintained.
  cls = super().__new__(mcls, name, bases, namespace, **kwargs)
2026-07-30 02:41:46,337: INFO: ---------------------------------------------------------------
2026-07-30 02:41:46,337: INFO: ---                 PROCESSING START                        ---
2026-07-30 02:41:46,337: INFO: ---------------------------------------------------------------
2026-07-30 02:41:46,337: INFO: conda environment path found: C:\Users\moses\miniconda3\envs\pyeo_env
2026-07-30 02:41:46,341: INFO: True
2026-07-30 02:41:46,343: INFO: ---------------------------------------------------------------
2026-07-30 02:41:46,343: INFO: ---                  INTEGRATED PROCESSING START            ---
2026-07-30 02:41:46,344: 

Libraries successfully imported
Required Amazon project paths: PASS
Configuration loaded: C:\GIS\projects\Amazon_BR163_PyEO\00_admin\amazon_br163_working.ini
PyEO directory: C:\GIS\src\pyeo
Critical configuration alignment: PASS


2026-07-30 02:41:46,543: WARNING: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'C:\GIS\projects\Amazon_BR163_PyEO\02_reference\sentinel2_tiles\sentinel2_tiles_br163.gpkg' may only be partially supported
2026-07-30 02:41:46,562: INFO: The provided ROI intersects with 2 Sentinel-2 tiles:
2026-07-30 02:41:46,562: INFO:   1 : 21MXM
2026-07-30 02:41:46,563: INFO:   2 : 21MYM
2026-07-30 02:41:46,579: INFO: ---------------------------------------------------------------
2026-07-30 02:41:46,581: INFO: ---                 PROCESSING START                        ---
2026-07-30 02:41:46,581: INFO: ---------------------------------------------------------------
2026-07-30 02:41:46,582: INFO: Amazon BR163 monitoring notebook initialised
2026-07-30 02:41:46,583: INFO: Selected pilot tile: 21MYM
2026-07-30 02:41:46,585: INFO: Copernicus Data Space Ecosystem is the download source
2026-07-30 02:41:46,586: INFO: Faulty Granule Threshold is set to: 350
2026-07-30 02:41:46,587: I

Tile-intersection output: C:\GIS\projects\Amazon_BR163_PyEO\01_roi\pilot\tilelist.csv
Selected pilot tile: 21MYM
Tile directory path: C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM
Folder structure build successfully finished
Active model: C:\GIS\projects\Amazon_BR163_PyEO\05_model\amazon_forest_clearing_extratrees.pkl
Configured from_classes: [1]
Configured to_classes: [2]
Forest-to-Clearing transition alignment: PASS
Credentials path: C:\GIS\secrets\pyeo_cdse.ini
Credentials configuration: PASS
Monitoring directory structure: PASS

AMAZON MONITORING NOTEBOOK INITIALISATION: PASS
Pilot tile: 21MYM
Configuration: C:\GIS\projects\Amazon_BR163_PyEO\00_admin\amazon_br163_working.ini
Download source: dataspace
Monitoring start: 20250701
Monitoring end: 20250930
Composite start: 20240701
Composite end: 20240930
CRS: 32721
Bands: ['B02', 'B03', 'B04', 'B08']
Output resolution: 10
Model: C:\GIS\projects\Amazon_BR163_PyEO\05_model\amazon_forest_clearing_extratrees.pkl
Transition: Forest class 1

In [4]:
print("do_all      :", config_dict["do_all"])
print("do_classify :", config_dict["do_classify"])
print("model_path  :", model_path)
print("output_dir  :", categorised_image_dir)

do_all      : False
do_classify : False
model_path  : C:\GIS\projects\Amazon_BR163_PyEO\05_model\amazon_forest_clearing_extratrees.pkl
output_dir  : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\output\classifications


# Download Sentinel-2 change detection images

In [5]:
tile_log.info(tile_to_process)

if config_dict["do_all"] or config_dict["do_download"]:

    tile_log.info(
        "---------------------------------------------------------------"
    )
    tile_log.info(
        "Querying change-detection images between "
        "{} and {} with cloud cover <= {}".format(
            start_date,
            end_date,
            cloud_cover,
        )
    )
    tile_log.info(
        "---------------------------------------------------------------"
    )

    if download_source == "dataspace":

        tiles_geom_path = os.path.join(
            config_dict["pyeo_dir"],
            config_dict["geometry_dir"],
            config_dict["s2_tiles_filename"],
        )

        if not os.path.isfile(tiles_geom_path):
            raise FileNotFoundError(
                "Sentinel-2 tile geometry file does not exist: "
                f"{tiles_geom_path}"
            )

        tiles_geom = gpd.read_file(
            os.path.abspath(tiles_geom_path)
        )

        if "Name" not in tiles_geom.columns:
            raise KeyError(
                "The Sentinel-2 tile geometry layer does not "
                "contain the required 'Name' field."
            )

        tile_geom = tiles_geom[
            tiles_geom["Name"] == tile_to_process
        ].copy()

        if tile_geom.empty:
            raise RuntimeError(
                f"Tile {tile_to_process} was not found in "
                f"{tiles_geom_path}"
            )

        tile_geom = tile_geom.to_crs(
            epsg=4326
        )

        geometry = tile_geom[
            "geometry"
        ].iloc[0].representative_point()

        # Convert configuration dates from YYYYMMDD
        # to the YYYY-MM-DD format required by CDSE.
        dataspace_change_start = datetime.strptime(
            start_date,
            "%Y%m%d",
        ).strftime(
            "%Y-%m-%d"
        )

        dataspace_change_end = datetime.strptime(
            end_date,
            "%Y%m%d",
        ).strftime(
            "%Y-%m-%d"
        )

        print("Querying Copernicus Data Space...")
        print("Tile:", tile_to_process)
        print("Start date:", dataspace_change_start)
        print("End date:", dataspace_change_end)
        print("Maximum cloud cover:", cloud_cover)

        # Do not hide the underlying query exception.
        dataspace_change_products_all = (
            queries_and_downloads.query_dataspace_by_tile_id(
                max_cloud_cover=cloud_cover,
                start_date=dataspace_change_start,
                end_date=dataspace_change_end,
                tile_id=tile_to_process,
                max_records=100,
                log=tile_log,
            )
        )

        if dataspace_change_products_all is None:
            raise RuntimeError(
                "The CDSE query returned None instead of "
                "a product table."
            )

        if not isinstance(
            dataspace_change_products_all,
            pd.DataFrame,
        ):
            raise TypeError(
                "The CDSE query returned an unexpected "
                "object type: "
                f"{type(dataspace_change_products_all)}"
            )

        if dataspace_change_products_all.empty:
            raise RuntimeError(
                "The CDSE query completed but returned no "
                "Sentinel-2 products for tile "
                f"{tile_to_process} between "
                f"{dataspace_change_start} and "
                f"{dataspace_change_end} with cloud cover "
                f"<= {cloud_cover}%."
            )

        print(
            "CDSE catalogue query returned",
            len(dataspace_change_products_all),
            "products.",
        )

        required_dataspace_columns = [
            "title",
            "relativeOrbitNumber",
            "processingLevel",
            "cloudCover",
            "startDate",
            "status",
        ]

        missing_dataspace_columns = [
            column
            for column in required_dataspace_columns
            if column
            not in dataspace_change_products_all.columns
        ]

        if missing_dataspace_columns:
            raise KeyError(
                "The CDSE query result is missing required "
                "columns: "
                f"{missing_dataspace_columns}\n"
                "Available columns: "
                f"{dataspace_change_products_all.columns.tolist()}"
            )

        titles = (
            dataspace_change_products_all[
                "title"
            ].tolist()
        )

        sizes = []
        uuids = []

        # The current adapter stores download metadata in
        # a nested dictionary column. Identify that column
        # safely rather than relying on tuple position.
        download_metadata_column = None

        for column_name in (
            dataspace_change_products_all.columns
        ):
            sample_values = (
                dataspace_change_products_all[
                    column_name
                ]
                .dropna()
                .head(3)
                .tolist()
            )

            if any(
                isinstance(value, dict)
                and "download" in value
                for value in sample_values
            ):
                download_metadata_column = column_name
                break

        if download_metadata_column is None:
            raise KeyError(
                "Could not identify the nested CDSE download "
                "metadata column.\n"
                "Available columns: "
                f"{dataspace_change_products_all.columns.tolist()}"
            )

        for metadata in (
            dataspace_change_products_all[
                download_metadata_column
            ]
        ):
            if not isinstance(metadata, dict):
                raise TypeError(
                    "Unexpected CDSE metadata value: "
                    f"{type(metadata)}"
                )

            download_info = metadata.get(
                "download",
                {}
            )

            size_bytes = download_info.get(
                "size"
            )

            download_url = download_info.get(
                "url"
            )

            if size_bytes is None:
                raise KeyError(
                    "A CDSE product is missing its "
                    "download size."
                )

            if not download_url:
                raise KeyError(
                    "A CDSE product is missing its "
                    "download URL."
                )

            sizes.append(
                size_bytes
            )

            uuids.append(
                str(download_url)
                .rstrip("/")
                .split("/")[-1]
            )

        relative_orbit_numbers = (
            dataspace_change_products_all[
                "relativeOrbitNumber"
            ].tolist()
        )

        processing_levels = (
            dataspace_change_products_all[
                "processingLevel"
            ].tolist()
        )

        transformed_levels = [
            (
                "Level-1C"
                if level == "S2MSI1C"
                else "Level-2A"
            )
            for level in processing_levels
        ]

        cloud_covers = (
            dataspace_change_products_all[
                "cloudCover"
            ].tolist()
        )

        begin_positions = (
            dataspace_change_products_all[
                "startDate"
            ].tolist()
        )

        statuses = (
            dataspace_change_products_all[
                "status"
            ].tolist()
        )

        scihub_compatible_df = pd.DataFrame(
            {
                "title": titles,
                "size": sizes,
                "beginposition": begin_positions,
                "relativeorbitnumber": (
                    relative_orbit_numbers
                ),
                "cloudcoverpercentage": cloud_covers,
                "processinglevel": transformed_levels,
                "uuid": uuids,
                "status": statuses,
            }
        )

        # Convert bytes to megabytes.
        scihub_compatible_df["size"] = (
            scihub_compatible_df[
                "size"
            ].apply(
                lambda value: round(
                    float(value) * 1e-6,
                    2,
                )
            )
        )

        df_all = scihub_compatible_df

    elif download_source == "scihub":

        raise RuntimeError(
            "SciHub is retired. The Amazon BR163 workflow "
            "must use download_source = dataspace."
        )

    else:

        raise RuntimeError(
            f"Unsupported download source: "
            f"{download_source}"
        )

    # Remove products below the configured size threshold.
    df = df_all.query(
        "size >= "
        + str(faulty_granule_threshold)
    ).copy()

    tile_log.info(
        "Removed {} faulty scenes <{} MB in size "
        "from the list.".format(
            len(df_all) - len(df),
            faulty_granule_threshold,
        )
    )

    df_faulty = df_all.query(
        "size < "
        + str(faulty_granule_threshold)
    )

    for row_index in range(
        len(df_faulty)
    ):
        tile_log.info(
            "   {} MB: {}".format(
                df_faulty.iloc[
                    row_index
                ]["size"],
                df_faulty.iloc[
                    row_index
                ]["title"],
            )
        )

    l1c_products = df[
        df["processinglevel"] == "Level-1C"
    ].copy()

    l2a_products = df[
        df["processinglevel"] == "Level-2A"
    ].copy()

    tile_log.info(
        "    {} L1C products".format(
            l1c_products.shape[0]
        )
    )

    tile_log.info(
        "    {} L2A products".format(
            l2a_products.shape[0]
        )
    )

    if (
        l1c_products.shape[0] > 0
        and l2a_products.shape[0] > 0
    ):

        tile_log.info(
            "Filtering L1C products that have the same "
            "acquisition timestamp as an existing "
            "L2A product."
        )

        l1c_products = (
            queries_and_downloads
            .filter_unique_dataspace_products(
                l1c_products=l1c_products,
                l2a_products=l2a_products,
                log=tile_log,
            )
        )

        tile_log.info(
            "--> {} L1C and L2A products with unique "
            "acquisition timestamps.".format(
                l1c_products.shape[0]
                + l2a_products.shape[0]
            )
        )

    tile_log.info(
        "%s L1C monitoring images",
        len(l1c_products),
    )

    tile_log.info(
        "%s L2A monitoring images",
        len(l2a_products),
    )

    print()
    print("MONITORING CATALOGUE QUERY: COMPLETE")
    print("All returned products:", len(df_all))
    print(
        "Eligible products after size filtering:",
        len(df),
    )
    print("L1C products:", len(l1c_products))
    print("L2A products:", len(l2a_products))

    display(
        df_all[
            [
                "title",
                "beginposition",
                "cloudcoverpercentage",
                "processinglevel",
                "size",
                "status",
                "uuid",
            ]
        ].sort_values(
            by=[
                "beginposition",
                "processinglevel",
            ]
        ).reset_index(
            drop=True
        )
    )

    tile_log.info(
        "Monitoring catalogue query cell "
        "successfully finished"
    )

else:
    print(
        "Monitoring catalogue query skipped because "
        "do_download=False and do_all=False."
    )

    # Define empty tables so later guarded display cells remain safe
    # during a validation-only Restart Kernel and Run All.
    df_all = pd.DataFrame()
    df = pd.DataFrame()
    l1c_products = pd.DataFrame()
    l2a_products = pd.DataFrame()


2026-07-30 02:41:46,658: INFO: 21MYM


Monitoring catalogue query skipped because do_download=False and do_all=False.


In [6]:
if config_dict["do_all"] or config_dict["do_download"]:
    print("All eligible products:", len(df_all))
    print("L1C products:", len(l1c_products))
    print("L2A products:", len(l2a_products))

    display(
        df_all[
            [
                "title",
                "beginposition",
                "cloudcoverpercentage",
                "processinglevel",
                "size",
                "status",
            ]
        ].sort_values(
            by=[
                "beginposition",
                "processinglevel",
            ]
        )
    )
else:
    print("Download catalogue inspection skipped because do_download=False and do_all=False.")


Download catalogue inspection skipped because do_download=False and do_all=False.


In [7]:
if config_dict["do_all"] or config_dict["do_download"]:
    print(l1c_products[["title", "beginposition"]])
    print(l2a_products[["title", "beginposition"]])
else:
    print("Download catalogue inspection skipped because do_download=False and do_all=False.")


Download catalogue inspection skipped because do_download=False and do_all=False.


In [8]:
if config_dict["do_all"] or config_dict["do_download"]:
    print("L1C count:", len(l1c_products))
    print("L2A count:", len(l2a_products))

    print()
    print(l1c_products[["title", "uuid", "processinglevel"]])

    print()
    print(l2a_products[["title", "uuid", "processinglevel"]].head())
else:
    print("Download catalogue inspection skipped because do_download=False and do_all=False.")


Download catalogue inspection skipped because do_download=False and do_all=False.


## Search for L2A Images Corresponding to L1C

In [9]:
if config_dict["do_all"] or config_dict["do_download"]:
    if download_source == "scihub":    
        if l1c_products.shape[0] > 0:
            tile_log.info("Checking for availability of L2A products to minimise download and atmospheric correction of L1C products.")
            n = len(l1c_products)
            drop = []
            add = []
            for r in range(n):
                id = l1c_products.iloc[r, :]["title"]
                search_term = (
                    "*"
                    + id.split("_")[2]
                    + "_"
                    + id.split("_")[3]
                    + "_"
                    + id.split("_")[4]
                    + "_"
                    + id.split("_")[5]
                    + "*"
                )
                tile_log.info("Search term: {}.".format(search_term))
                matching_l2a_products = queries_and_downloads._file_api_query(
                    user=sen_user,
                    passwd=sen_pass,
                    start_date=start_date,
                    end_date=end_date,
                    filename=search_term,
                    cloud=cloud_cover,
                    producttype="S2MSI2A",
                )

                matching_l2a_products_df = pd.DataFrame.from_dict(
                    matching_l2a_products, orient="index"
                )
                if len(matching_l2a_products_df) == 1:
                    tile_log.info(matching_l2a_products_df.iloc[0, :]["size"])
                    matching_l2a_products_df["size"] = (
                        matching_l2a_products_df["size"]
                        .str.split(" ")
                        .apply(
                            lambda x: float(x[0])
                            * {"GB": 1e3, "MB": 1, "KB": 1e-3}[x[1]]
                        )
                    )
                    if (
                        matching_l2a_products_df.iloc[0, :]["size"]
                        > faulty_granule_threshold
                    ):
                        tile_log.info("Replacing L1C {} with L2A product:".format(id))
                        tile_log.info(
                            "              {}".format(
                                matching_l2a_products_df.iloc[0, :]["title"]
                            )
                        )
                        drop.append(l1c_products.index[r])
                        add.append(matching_l2a_products_df.iloc[0, :])
                if len(matching_l2a_products_df) == 0:
                    tile_log.info("Found no match for L1C: {}.".format(id))
                if len(matching_l2a_products_df) > 1:
                    # check granule sizes on the server
                    matching_l2a_products_df["size"] = (
                        matching_l2a_products_df["size"]
                        .str.split(" ")
                        .apply(
                            lambda x: float(x[0])
                            * {"GB": 1e3, "MB": 1, "KB": 1e-3}[x[1]]
                        )
                    )
                    if (
                        matching_l2a_products_df.iloc[0, :]["size"]
                        > faulty_granule_threshold
                    ):
                        tile_log.info("Replacing L1C {} with L2A product:".format(id))
                        tile_log.info(
                            "              {}".format(
                                matching_l2a_products_df.iloc[0, :]["title"]
                            )
                        )
                        drop.append(l1c_products.index[r])
                        add.append(matching_l2a_products_df.iloc[0, :])

            if len(drop) > 0:
                l1c_products = l1c_products.drop(index=drop)
            if len(add) > 0:
                if config_dict["do_dev"]:
                    add = pd.DataFrame(add)
                    l2a_products = pd.concat([l2a_products, add])
                else:
                    add = pd.DataFrame(add)
                    l2a_products = pd.concat([l2a_products, add])

            tile_log.info(
                "    {} L1C products remaining for download".format(
                    l1c_products.shape[0]
                )
            )
    l2a_products = l2a_products.drop_duplicates(subset="title")
    tile_log.info("    {} L2A products remaining for download".format(l2a_products.shape[0]))

## Download and Pre-Process L1C Change Imagery

- If there any L1C products in the change images search query that are not matched with L2A products, then download these L1Cs and apply `atmospheric_correction`.

In [10]:
if config_dict["do_all"] or config_dict["do_download"]:
    if l1c_products.shape[0] > 0:

        tile_log.info(f"Downloading Sentinel-2 L1C products from {download_source}")

        if download_source == "scihub":
            queries_and_downloads.download_s2_data_from_df(
                l1c_products,
                l1_image_dir,
                l2_image_dir,
                download_source,
                user=sen_user,
                passwd=sen_pass,
                try_scihub_on_fail=True,
            )
        elif download_source == "dataspace":
                queries_and_downloads.download_s2_data_from_dataspace(
                product_df=l1c_products,
                l1c_directory=l1_image_dir,
                l2a_directory=l2_image_dir,
                dataspace_username=sen_user,
                dataspace_password=sen_pass,
                log=tile_log
            )
        else:
            tile_log.error(f"download source specified did not match 'scihub' or 'dataspace'")
            tile_log.error(f"download source supplied was  :  {download_source}")
            tile_log.error("exiting pipeline...")
            sys.exit(1)

    #     tile_log.info("Atmospheric correction with sen2cor.")
    #     raster_manipulation.atmospheric_correction(
    #         l1_image_dir,
    #         l2_image_dir,
    #         sen2cor_path,
    #         delete_unprocessed_image=False,
    #         log=tile_log,
    #     )

    tile_log.info("Successfully downloaded the Sentinel-2 L1C products")

## Download L2A Change Imagery

## Sentinel-2 L2A Download, Recovery, and Filesystem QA

This section downloads the expected monitoring-period Sentinel-2 L2A products, skips products already present, performs end-of-pass filesystem QA, retries only missing products, and returns a final PASS or FAIL result.

Validated on 28 July 2026:
- Expected products: 19
- Valid SAFE folders: 19
- Missing products: 0
- Incomplete SAFE directories: 0
- Retry passes used: 2
- Final status: PASS

In [11]:
if config_dict["do_all"] or config_dict["do_download"]:
    # Sentinel-2 L2A download with end-of-pass QA and controlled retries

    import os
    import shutil
    import time


    # ---------------------------------------------------------------------
    # USER-CONTROLLABLE RETRY SETTINGS
    # ---------------------------------------------------------------------

    # Number of additional retry passes after the initial full pass.
    # Initial pass + 3 retries = maximum of 4 total attempts per missing product.
    max_retry_passes = 3

    # Pause between complete passes, not between individual products.
    retry_wait_seconds = 15


    # ---------------------------------------------------------------------
    # HELPER FUNCTIONS
    # ---------------------------------------------------------------------

    def normalise_safe_name(product_title):
        """
        Return the expected SAFE directory name for a Sentinel-2 product.
        """
        product_title = str(product_title).strip()

        if product_title.endswith(".SAFE"):
            return product_title

        return f"{product_title}.SAFE"


    def get_expected_product_inventory(product_df):
        """
        Build a dictionary linking expected SAFE directory names to product titles.
        """
        if "title" not in product_df.columns:
            raise KeyError(
                "The L2A product dataframe does not contain the required "
                "'title' column."
            )

        inventory = {}

        for product_title in product_df["title"].astype(str):
            safe_name = normalise_safe_name(product_title)
            inventory[safe_name] = product_title

        return inventory


    def get_present_expected_safe_names(expected_safe_names, directory):
        """
        Return expected SAFE directory names that currently exist.
        """
        return {
            safe_name
            for safe_name in expected_safe_names
            if os.path.isdir(os.path.join(directory, safe_name))
        }


    def detect_expected_incomplete_safe_dirs(
        expected_safe_names,
        directory,
        threshold_mb,
    ):
        """
        Detect undersized SAFE directories, limited to products expected
        during the current download run.

        Returns:
            list of dictionaries containing:
            - safe_name
            - safe_path
            - size_bytes
            - size_mb
        """
        incomplete_downloads, sizes = raster_manipulation.find_small_safe_dirs(
            directory,
            threshold=threshold_mb * 1024 * 1024,
        )

        incomplete_expected = []

        for safe_dir, size_bytes in zip(incomplete_downloads, sizes):
            safe_name = os.path.basename(os.path.normpath(safe_dir))
            size_mb = size_bytes / 1024 / 1024

            if (
                safe_name in expected_safe_names
                and os.path.isdir(safe_dir)
                and size_mb < threshold_mb
            ):
                incomplete_expected.append(
                    {
                        "safe_name": safe_name,
                        "safe_path": safe_dir,
                        "size_bytes": size_bytes,
                        "size_mb": size_mb,
                    }
                )

        return incomplete_expected


    def remove_expected_incomplete_safe_dirs(incomplete_safe_dirs):
        """
        Remove only expected SAFE directories that were identified as
        undersized and likely incomplete.

        Removal is necessary because the downloader skips products whenever
        a SAFE directory with the expected name already exists.
        """
        for item in incomplete_safe_dirs:
            safe_path = item["safe_path"]
            safe_name = item["safe_name"]
            size_mb = item["size_mb"]

            tile_log.warning(
                "Removing likely incomplete SAFE directory before retry: "
                f"{safe_name} ({size_mb:.1f} MB)"
            )

            try:
                shutil.rmtree(safe_path)

                tile_log.info(
                    f"Removed incomplete SAFE directory: {safe_path}"
                )

            except Exception as exc:
                tile_log.error(
                    "Could not remove incomplete SAFE directory: "
                    f"{safe_path}"
                )
                tile_log.error(f"Removal error: {exc}")

                raise RuntimeError(
                    "Download recovery cannot continue because an incomplete "
                    f"SAFE directory could not be removed: {safe_path}"
                ) from exc


    def build_missing_product_dataframe(
        full_product_df,
        expected_inventory,
        missing_safe_names,
    ):
        """
        Return only dataframe rows corresponding to missing SAFE products.
        """
        missing_titles = {
            expected_inventory[safe_name]
            for safe_name in missing_safe_names
        }

        return full_product_df[
            full_product_df["title"].astype(str).isin(missing_titles)
        ].copy()


    def run_l2a_download_pass(product_df, pass_number, pass_label):
        """
        Run one complete download pass for the supplied product dataframe.
        No product is immediately retried within this function.
        """
        product_count = product_df.shape[0]

        tile_log.info("=" * 79)
        tile_log.info(
            f"{pass_label} {pass_number}: attempting {product_count} "
            "Sentinel-2 L2A product(s)"
        )
        tile_log.info("=" * 79)

        if product_count == 0:
            tile_log.info(
                "No products require downloading during this pass."
            )
            return

        if download_source == "scihub":
            queries_and_downloads.download_s2_data(
                product_df.to_dict("index"),
                l1_image_dir,
                l2_image_dir,
                download_source,
                user=sen_user,
                passwd=sen_pass,
                try_scihub_on_fail=True,
            )

        elif download_source == "dataspace":
            queries_and_downloads.download_s2_data_from_dataspace(
                product_df=product_df,
                l1c_directory=l1_image_dir,
                l2a_directory=l2_image_dir,
                dataspace_username=sen_user,
                dataspace_password=sen_pass,
                log=tile_log,
            )

        else:
            raise ValueError(
                "Unsupported download source: "
                f"{download_source!r}. Expected 'scihub' or 'dataspace'."
            )


    def perform_download_qa(
        expected_inventory,
        directory,
        threshold_mb,
    ):
        """
        Inspect the filesystem after a complete pass.

        Returns:
            present_safe_names
            missing_safe_names
            incomplete_safe_dirs
        """
        expected_safe_names = set(expected_inventory.keys())

        present_safe_names = get_present_expected_safe_names(
            expected_safe_names,
            directory,
        )

        incomplete_safe_dirs = detect_expected_incomplete_safe_dirs(
            expected_safe_names,
            directory,
            threshold_mb,
        )

        incomplete_safe_names = {
            item["safe_name"]
            for item in incomplete_safe_dirs
        }

        # Incomplete SAFE directories must not count as valid downloads.
        valid_present_safe_names = (
            present_safe_names - incomplete_safe_names
        )

        missing_safe_names = (
            expected_safe_names - valid_present_safe_names
        )

        return (
            valid_present_safe_names,
            missing_safe_names,
            incomplete_safe_dirs,
        )


    def log_pass_qa_summary(
        pass_name,
        expected_count,
        present_count,
        missing_safe_names,
        incomplete_safe_dirs,
    ):
        """
        Print a consistent QA summary after each complete pass.
        """
        tile_log.info("-" * 79)
        tile_log.info(f"{pass_name} QA SUMMARY")
        tile_log.info("-" * 79)
        tile_log.info(f"Expected products           : {expected_count}")
        tile_log.info(f"Valid SAFE folders present  : {present_count}")
        tile_log.info(
            f"Missing products            : {len(missing_safe_names)}"
        )
        tile_log.info(
            "Incomplete SAFE directories : "
            f"{len(incomplete_safe_dirs)}"
        )

        if missing_safe_names:
            tile_log.warning(
                "Products requiring another download attempt:"
            )

            for safe_name in sorted(missing_safe_names):
                tile_log.warning(f"    {safe_name}")

        if incomplete_safe_dirs:
            tile_log.warning(
                "Expected SAFE directories identified as incomplete:"
            )

            for item in sorted(
                incomplete_safe_dirs,
                key=lambda value: value["safe_name"],
            ):
                tile_log.warning(
                    f"    {item['safe_name']} "
                    f"({item['size_mb']:.1f} MB)"
                )

        tile_log.info("-" * 79)


    # ---------------------------------------------------------------------
    # MAIN DOWNLOAD AND RECOVERY WORKFLOW
    # ---------------------------------------------------------------------

    if config_dict["do_all"] or config_dict["do_download"]:

        os.makedirs(l2_image_dir, exist_ok=True)

        if l2a_products.shape[0] == 0:
            tile_log.warning(
                "No Sentinel-2 L2A products were available for download."
            )

            tile_log.info("-" * 79)
            tile_log.info(
                "Image acquisition stage finished with no L2A products "
                "selected."
            )
            tile_log.info("-" * 79)

        else:
            expected_inventory = get_expected_product_inventory(
                l2a_products
            )

            expected_safe_names = set(expected_inventory.keys())
            expected_count = len(expected_safe_names)

            tile_log.info("=" * 79)
            tile_log.info(
                "SENTINEL-2 L2A DOWNLOAD AND RECOVERY WORKFLOW"
            )
            tile_log.info("=" * 79)
            tile_log.info(f"Download source             : {download_source}")
            tile_log.info(f"Expected products           : {expected_count}")
            tile_log.info(
                f"Additional retry passes     : {max_retry_passes}"
            )
            tile_log.info(
                f"Retry wait between passes   : "
                f"{retry_wait_seconds} second(s)"
            )
            tile_log.info(f"L2A output directory        : {l2_image_dir}")
            tile_log.info("=" * 79)

            # -------------------------------------------------------------
            # INITIAL FULL PASS
            # -------------------------------------------------------------

            run_l2a_download_pass(
                product_df=l2a_products,
                pass_number=1,
                pass_label="INITIAL DOWNLOAD PASS",
            )

            (
                valid_present_safe_names,
                missing_safe_names,
                incomplete_safe_dirs,
            ) = perform_download_qa(
                expected_inventory=expected_inventory,
                directory=l2_image_dir,
                threshold_mb=faulty_granule_threshold,
            )

            log_pass_qa_summary(
                pass_name="INITIAL DOWNLOAD PASS",
                expected_count=expected_count,
                present_count=len(valid_present_safe_names),
                missing_safe_names=missing_safe_names,
                incomplete_safe_dirs=incomplete_safe_dirs,
            )

            # Remove incomplete expected directories only after the complete
            # initial pass has ended and QA has identified them.
            if incomplete_safe_dirs:
                remove_expected_incomplete_safe_dirs(
                    incomplete_safe_dirs
                )

                (
                    valid_present_safe_names,
                    missing_safe_names,
                    incomplete_safe_dirs,
                ) = perform_download_qa(
                    expected_inventory=expected_inventory,
                    directory=l2_image_dir,
                    threshold_mb=faulty_granule_threshold,
                )

            retry_passes_used = 0

            # -------------------------------------------------------------
            # END-OF-PASS RETRIES
            # -------------------------------------------------------------

            for retry_number in range(1, max_retry_passes + 1):

                if not missing_safe_names:
                    break

                retry_passes_used = retry_number

                tile_log.warning(
                    f"Initial/post-retry QA found "
                    f"{len(missing_safe_names)} missing product(s)."
                )

                tile_log.info(
                    f"Waiting {retry_wait_seconds} second(s) before "
                    f"retry pass {retry_number}."
                )

                time.sleep(retry_wait_seconds)

                missing_product_df = build_missing_product_dataframe(
                    full_product_df=l2a_products,
                    expected_inventory=expected_inventory,
                    missing_safe_names=missing_safe_names,
                )

                run_l2a_download_pass(
                    product_df=missing_product_df,
                    pass_number=retry_number,
                    pass_label="RETRY PASS",
                )

                (
                    valid_present_safe_names,
                    missing_safe_names,
                    incomplete_safe_dirs,
                ) = perform_download_qa(
                    expected_inventory=expected_inventory,
                    directory=l2_image_dir,
                    threshold_mb=faulty_granule_threshold,
                )

                log_pass_qa_summary(
                    pass_name=f"RETRY PASS {retry_number}",
                    expected_count=expected_count,
                    present_count=len(valid_present_safe_names),
                    missing_safe_names=missing_safe_names,
                    incomplete_safe_dirs=incomplete_safe_dirs,
                )

                # Removal happens only after the complete retry pass has
                # finished. Removed products remain in the missing list and
                # become eligible for the next complete retry pass.
                if incomplete_safe_dirs:
                    remove_expected_incomplete_safe_dirs(
                        incomplete_safe_dirs
                    )

                    (
                        valid_present_safe_names,
                        missing_safe_names,
                        incomplete_safe_dirs,
                    ) = perform_download_qa(
                        expected_inventory=expected_inventory,
                        directory=l2_image_dir,
                        threshold_mb=faulty_granule_threshold,
                    )

            # -------------------------------------------------------------
            # FINAL FILESYSTEM QA
            # -------------------------------------------------------------

            (
                valid_present_safe_names,
                missing_safe_names,
                incomplete_safe_dirs,
            ) = perform_download_qa(
                expected_inventory=expected_inventory,
                directory=l2_image_dir,
                threshold_mb=faulty_granule_threshold,
            )

            tile_log.info("=" * 79)
            tile_log.info("FINAL SENTINEL-2 L2A DOWNLOAD QA")
            tile_log.info("=" * 79)
            tile_log.info(f"Expected products           : {expected_count}")
            tile_log.info(
                "Valid SAFE folders present  : "
                f"{len(valid_present_safe_names)}"
            )
            tile_log.info(
                f"Missing products            : "
                f"{len(missing_safe_names)}"
            )
            tile_log.info(
                "Incomplete SAFE directories : "
                f"{len(incomplete_safe_dirs)}"
            )
            tile_log.info(
                f"Retry passes used           : {retry_passes_used}"
            )

            if not missing_safe_names and not incomplete_safe_dirs:
                tile_log.info("")
                tile_log.info("STATUS: PASS")
                tile_log.info(
                    "All expected Sentinel-2 L2A products are present, "
                    "and no incomplete SAFE directories were detected."
                )
                tile_log.info("")
                tile_log.info(
                    "Image download and atmospheric correction for "
                    "change detection images is complete."
                )
                tile_log.info("=" * 79)

            else:
                tile_log.error("")
                tile_log.error("STATUS: FAIL")
                tile_log.error(
                    "The Sentinel-2 L2A dataset remains incomplete."
                )

                if missing_safe_names:
                    tile_log.error("")
                    tile_log.error(
                        "Products still missing after all download passes:"
                    )

                    for safe_name in sorted(missing_safe_names):
                        tile_log.error(f"    {safe_name}")

                if incomplete_safe_dirs:
                    tile_log.error("")
                    tile_log.error(
                        "Incomplete SAFE directories still present:"
                    )

                    for item in sorted(
                        incomplete_safe_dirs,
                        key=lambda value: value["safe_name"],
                    ):
                        tile_log.error(
                            f"    {item['safe_name']} "
                            f"({item['size_mb']:.1f} MB)"
                        )

                tile_log.error("")
                tile_log.error(
                    "Do not continue to change detection until the "
                    "download QA returns STATUS: PASS."
                )
                tile_log.error("=" * 79)

                raise RuntimeError(
                    "Sentinel-2 L2A download QA failed. "
                    f"{len(missing_safe_names)} product(s) remain missing "
                    f"and {len(incomplete_safe_dirs)} incomplete SAFE "
                    "director(ies) remain."
                )
else:
    print(
        "Sentinel-2 download and retry stage skipped because "
        "do_download=False and do_all=False."
    )


Sentinel-2 download and retry stage skipped because do_download=False and do_all=False.


In [12]:

# Sentinel-2 L2A filesystem inventory
#
# This cell is operational in both modes:
# - processing mode: reports the products downloaded by this run;
# - validation mode: inspects existing SAFE folders without requiring downloads.

from pathlib import Path

l2a_path = Path(l2_image_dir)
safe_folders = (
    sorted(
        path for path in l2a_path.iterdir()
        if path.is_dir() and path.name.endswith(".SAFE")
    )
    if l2a_path.is_dir()
    else []
)

download_enabled = bool(
    config_dict["do_all"] or config_dict["do_download"]
)

print("=" * 79)
print("FINAL L2A FILESYSTEM INVENTORY")
print("=" * 79)
print(f"Download stage enabled : {download_enabled}")
print(f"L2A directory          : {l2a_path}")
print(f"SAFE folders           : {len(safe_folders)}")
print()

for index, safe_folder in enumerate(safe_folders, start=1):
    print(f"{index:02d}. {safe_folder.name}")

print("=" * 79)

if download_enabled:
    # Cell 22 performs the authoritative missing/incomplete-product
    # checks and raises if the enabled download stage is incomplete.
    print(
        "STATUS: PASS\n"
        "The enabled download stage completed its controlled QA. "
        f"{len(safe_folders)} SAFE folder(s) are currently present."
    )
else:
    print(
        "STATUS: INFORMATIONAL\n"
        "Download is disabled. Existing SAFE folders were inventoried "
        "without querying or writing data."
    )


FINAL L2A FILESYSTEM INVENTORY
Download stage enabled : False
L2A directory          : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\L2A
SAFE folders           : 19

01. S2A_MSIL2A_20250706T140111_N0511_R067_T21MYM_20250706T170223.SAFE
02. S2A_MSIL2A_20250716T140111_N0511_R067_T21MYM_20250716T184813.SAFE
03. S2A_MSIL2A_20250726T140111_N0511_R067_T21MYM_20250726T183013.SAFE
04. S2A_MSIL2A_20250805T140111_N0511_R067_T21MYM_20250805T190614.SAFE
05. S2A_MSIL2A_20250825T140111_N0511_R067_T21MYM_20250825T183615.SAFE
06. S2A_MSIL2A_20250924T140111_N0511_R067_T21MYM_20250924T183959.SAFE
07. S2B_MSIL2A_20250709T135709_N0511_R067_T21MYM_20250709T203814.SAFE
08. S2B_MSIL2A_20250719T135709_N0511_R067_T21MYM_20250719T172719.SAFE
09. S2B_MSIL2A_20250729T135659_N0511_R067_T21MYM_20250729T192603.SAFE
10. S2B_MSIL2A_20250818T135709_N0511_R067_T21MYM_20250818T193153.SAFE
11. S2B_MSIL2A_20250907T135659_N0511_R067_T21MYM_20250907T191708.SAFE
12. S2B_MSIL2A_20250927T135659_N0511_R067_T21MYM_20250927

In [13]:

# Validate the internal structure of the Sentinel-2 L2A SAFE products
# currently present. Missing products are handled by the enabled download
# stage; this cell validates the internal structure of folders that exist.

from pathlib import Path

structure_results = []
invalid_safe_folders = []

for safe_folder in safe_folders:
    missing_items = []

    manifest_path = safe_folder / "manifest.safe"
    granule_path = safe_folder / "GRANULE"

    if not manifest_path.is_file():
        missing_items.append("manifest.safe")

    if not granule_path.is_dir():
        missing_items.append("GRANULE")

    img_data_directories = (
        list(granule_path.glob("*/IMG_DATA"))
        if granule_path.is_dir()
        else []
    )

    if not img_data_directories:
        missing_items.append("GRANULE/*/IMG_DATA")

    jp2_files = []
    for img_data_directory in img_data_directories:
        jp2_files.extend(img_data_directory.rglob("*.jp2"))

    if not jp2_files:
        missing_items.append("JP2 imagery")

    valid = not missing_items
    structure_results.append(
        {
            "safe_folder": safe_folder.name,
            "valid": valid,
            "missing": missing_items,
            "jp2_count": len(jp2_files),
        }
    )

    if not valid:
        invalid_safe_folders.append(
            {
                "safe_folder": safe_folder.name,
                "missing": missing_items,
            }
        )

print("=" * 79)
print("L2A SAFE STRUCTURE QA")
print("=" * 79)
print(f"SAFE folders checked : {len(structure_results)}")
print(
    "Valid SAFE folders   :",
    len(structure_results) - len(invalid_safe_folders),
)
print(f"Invalid SAFE folders : {len(invalid_safe_folders)}")

for result in structure_results:
    status = "PASS" if result["valid"] else "FAIL"
    print(
        f"{status}: {result['safe_folder']} "
        f"(JP2 files: {result['jp2_count']})"
    )
    if result["missing"]:
        print("       Missing:", ", ".join(result["missing"]))

strict_safe_qa = bool(
    config_dict["do_all"]
    or config_dict["do_download"]
    or config_dict["do_change"]
    or config_dict["do_classify"]
)

if invalid_safe_folders and strict_safe_qa:
    raise RuntimeError(
        "SAFE structure QA failed. One or more existing products "
        "is incomplete. Repair or remove the incomplete product "
        "before enabled downstream processing."
    )
elif invalid_safe_folders:
    print(
        "STATUS: WARNING\n"
        "Incomplete SAFE folders exist, but all dependent processing "
        "stages are disabled. No processing was attempted."
    )

if not structure_results:
    print(
        "STATUS: SKIPPED\n"
        "No SAFE folders are present. This is acceptable only while "
        "download and downstream processing stages remain disabled."
    )
else:
    print(
        "STATUS: PASS\n"
        "All existing SAFE folders contain manifest.safe, GRANULE, "
        "IMG_DATA, and JP2 imagery."
    )


L2A SAFE STRUCTURE QA
SAFE folders checked : 19
Valid SAFE folders   : 19
Invalid SAFE folders : 0
PASS: S2A_MSIL2A_20250706T140111_N0511_R067_T21MYM_20250706T170223.SAFE (JP2 files: 36)
PASS: S2A_MSIL2A_20250716T140111_N0511_R067_T21MYM_20250716T184813.SAFE (JP2 files: 36)
PASS: S2A_MSIL2A_20250726T140111_N0511_R067_T21MYM_20250726T183013.SAFE (JP2 files: 36)
PASS: S2A_MSIL2A_20250805T140111_N0511_R067_T21MYM_20250805T190614.SAFE (JP2 files: 36)
PASS: S2A_MSIL2A_20250825T140111_N0511_R067_T21MYM_20250825T183615.SAFE (JP2 files: 36)
PASS: S2A_MSIL2A_20250924T140111_N0511_R067_T21MYM_20250924T183959.SAFE (JP2 files: 36)
PASS: S2B_MSIL2A_20250709T135709_N0511_R067_T21MYM_20250709T203814.SAFE (JP2 files: 36)
PASS: S2B_MSIL2A_20250719T135709_N0511_R067_T21MYM_20250719T172719.SAFE (JP2 files: 36)
PASS: S2B_MSIL2A_20250729T135659_N0511_R067_T21MYM_20250729T192603.SAFE (JP2 files: 36)
PASS: S2B_MSIL2A_20250818T135709_N0511_R067_T21MYM_20250818T193153.SAFE (JP2 files: 36)
PASS: S2B_MSIL2A_2025

## Housekeeping - Compress L1Cs

If you have set your `do_zip` argument to `True`, then this cell will compress the L1Cs now that they have been atmospherically corrected and relabelled as L2As.

In [14]:
if config_dict["do_all"] or config_dict["do_download"]:
    if config_dict["do_delete"]:
        tile_log.info("---------------------------------------------------------------")
        tile_log.info("Deleting L1C images downloaded for change detection.")
        tile_log.info("Keeping only the derived L2A images after atmospheric correction.")
        tile_log.info("---------------------------------------------------------------")
        directory = l1_image_dir
        tile_log.info("Deleting {}".format(directory))
        shutil.rmtree(directory)
        tile_log.info("---------------------------------------------------------------")
        tile_log.info("Deletion complete")
        tile_log.info("---------------------------------------------------------------")
    else:
        if config_dict["do_zip"]:
            tile_log.info("---------------------------------------------------------------")
            tile_log.info("Zipping L1C images downloaded for change detection")
            tile_log.info("---------------------------------------------------------------")
            filesystem_utilities.zip_contents(l1_image_dir)
            tile_log.info("---------------------------------------------------------------")
            tile_log.info("Zipping complete")
            tile_log.info("---------------------------------------------------------------")

    tile_log.info("Cell successfully finished")

# Cloud Masking, Offsetting and Quicklooks

Here, like before in the previous session, we cloud mask, apply the baseline offset correction and produce quicklooks (if selected).  

Additionally, if you have set the `do_zip` flag to True, then `pyeo` will compress the cloud masked L2A images, as we no longer need these once classified.

## Cloud Masking

In [15]:

# Cloud-masking stage preflight
#
# Strict input validation runs only when change processing is enabled.
# In validation-only mode the cell inventories existing inputs/outputs
# and skips without creating directories or processing rasters.

from pathlib import Path

source_l2a_dir = Path(l2_image_dir)
masked_output_dir = Path(l2_masked_image_dir)

safe_products = (
    sorted(source_l2a_dir.glob("*.SAFE"))
    if source_l2a_dir.is_dir()
    else []
)
existing_masked_tifs = (
    sorted(masked_output_dir.glob("*.tif"))
    if masked_output_dir.is_dir()
    else []
)

cloud_masking_enabled = bool(
    config_dict["do_all"] or config_dict["do_change"]
)

print("=" * 79)
print("CLOUD-MASKING PREFLIGHT")
print("=" * 79)
print(f"Stage enabled               : {cloud_masking_enabled}")
print(f"Source L2A directory        : {source_l2a_dir}")
print(f"Source SAFE products        : {len(safe_products)}")
print(f"Masked output directory     : {masked_output_dir}")
print(f"Existing masked TIFF files  : {len(existing_masked_tifs)}")
print("=" * 79)

if cloud_masking_enabled:
    if not source_l2a_dir.is_dir():
        raise FileNotFoundError(
            f"Source L2A directory does not exist:\n{source_l2a_dir}"
        )

    if not safe_products:
        raise RuntimeError(
            "Cloud masking is enabled, but no L2A SAFE products "
            "were found. Enable and complete the download stage first."
        )

    masked_output_dir.mkdir(parents=True, exist_ok=True)

    print("STATUS: PASS")
    print("Cloud-masking inputs and output directory are ready.")
else:
    print("STATUS: SKIPPED")
    print(
        "Cloud masking is disabled. Existing files were inspected "
        "read-only and no directory or raster was created."
    )


CLOUD-MASKING PREFLIGHT
Stage enabled               : False
Source L2A directory        : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\L2A
Source SAFE products        : 19
Masked output directory     : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\cloud_masked
Existing masked TIFF files  : 19
STATUS: SKIPPED
Cloud masking is disabled. Existing files were inspected read-only and no directory or raster was created.


In [16]:
if config_dict["do_all"] or config_dict["do_change"]:
    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Applying simple cloud, cloud shadow and haze mask based on SCL files and stacking the masked band raster files."
    )
    tile_log.info("---------------------------------------------------------------")

    directory = l2_masked_image_dir
    masked_file_paths = [
        f
        for f in os.listdir(directory)
        if f.endswith(".tif") and os.path.isfile(os.path.join(directory, f))
    ]

    directory = l2_image_dir
    l2a_zip_file_paths = [f for f in os.listdir(directory) if f.endswith(".zip")]

    if len(l2a_zip_file_paths) > 0:
        for f in l2a_zip_file_paths:
            # check whether the zipped file has already been cloud masked
            zip_timestamp = filesystem_utilities.get_image_acquisition_time(
                os.path.basename(f)
            ).strftime("%Y%m%dT%H%M%S")
            if any(zip_timestamp in f for f in masked_file_paths):
                continue
            else:
                # extract it if not
                filesystem_utilities.unzip_contents(
                    os.path.join(composite_l2_image_dir, f),
                    ifstartswith="S2",
                    ending=".SAFE",
                )

    directory = l2_image_dir
    l2a_safe_file_paths = [
        f
        for f in os.listdir(directory)
        if f.endswith(".SAFE") and os.path.isdir(os.path.join(directory, f))
    ]

    files_for_cloud_masking = []
    if len(l2a_safe_file_paths) > 0:
        for f in l2a_safe_file_paths:
            # check whether the L2A SAFE file has already been cloud masked
            safe_timestamp = filesystem_utilities.get_image_acquisition_time(
                os.path.basename(f)
            ).strftime("%Y%m%dT%H%M%S")
            if any(safe_timestamp in f for f in masked_file_paths):
                continue
            else:
                # add it to the list of files to do if it has not been cloud masked yet
                files_for_cloud_masking = files_for_cloud_masking + [f]
    
    if len(files_for_cloud_masking) == 0:
        tile_log.info("No L2A images found for cloud masking. They may already have been done.")
    else:
        tile_log.info("Files for cloud masking:")
        for f in files_for_cloud_masking:
            tile_log.info(f"  {f}")
        raster_manipulation.apply_scl_cloud_mask(
            l2_image_dir,
            l2_masked_image_dir,
            scl_classes=[0, 1, 2, 3, 8, 9, 10, 11],
            buffer_size=buffer_size,
            bands=bands,
            out_resolution=out_resolution,
            haze=None,
            epsg=epsg,
            skip_existing=skip_existing,
            log=tile_log,
            )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Cloud masking and band stacking of new L2A images are complete.")
    tile_log.info("---------------------------------------------------------------")

## QA Gate: Cloud-Masked Monitoring Images

This QA gate verifies that all expected monitoring-period L2A scenes produced complete cloud-masked GeoTIFF outputs before baseline-offset correction or change detection continues.

Required result:
- 19 expected L2A scenes
- 19 cloud-masked TIFF outputs
- No missing scenes
- No unexpected duplicate acquisition timestamps
- All outputs readable by GDAL
- Four raster bands per output
- EPSG:32721
- Consistent raster dimensions
- No zero-byte or abnormally small files
- No residual PyEO temporary directories

In [17]:

# ============================================================
# QA GATE: CLOUD-MASKED MONITORING IMAGES
# ============================================================
#
# When do_change/do_all is enabled, this is a strict gate.
# When disabled, it is a read-only inventory and never blocks Run All.

from pathlib import Path
from osgeo import gdal

l2a_directory = Path(l2_image_dir)
cloud_masked_directory = Path(l2_masked_image_dir)

source_safe_products = (
    sorted(l2a_directory.glob("*.SAFE"))
    if l2a_directory.is_dir()
    else []
)
masked_tifs = (
    sorted(cloud_masked_directory.glob("*.tif"))
    if cloud_masked_directory.is_dir()
    else []
)

change_enabled = bool(
    config_dict["do_all"] or config_dict["do_change"]
)

print("=" * 78)
print("CLOUD-MASKED MONITORING IMAGE QA")
print("=" * 78)
print(f"Change stage enabled      : {change_enabled}")
print(f"L2A source directory      : {l2a_directory}")
print(f"Source SAFE products      : {len(source_safe_products)}")
print(f"Cloud-masked directory    : {cloud_masked_directory}")
print(f"Cloud-masked TIFF files   : {len(masked_tifs)}")

readable_count = 0
unreadable_files = []

for tif_path in masked_tifs:
    dataset = gdal.Open(str(tif_path))
    if dataset is None:
        unreadable_files.append(tif_path.name)
    else:
        readable_count += 1
    dataset = None

print(f"Readable TIFF files       : {readable_count}")
print(f"Unreadable TIFF files     : {len(unreadable_files)}")

if unreadable_files:
    print("Unreadable files:")
    for filename in unreadable_files:
        print("  -", filename)

if change_enabled:
    if not source_safe_products:
        raise RuntimeError(
            "Change processing is enabled, but no source SAFE "
            "products were found."
        )

    if len(masked_tifs) != len(source_safe_products):
        raise RuntimeError(
            "Cloud-masked output count does not match the source "
            f"SAFE count ({len(masked_tifs)} versus "
            f"{len(source_safe_products)})."
        )

    if unreadable_files:
        raise RuntimeError(
            "One or more cloud-masked TIFF files cannot be opened."
        )

    print("QA STATUS: PASS")
    print("Cloud-masked monitoring images are ready.")
else:
    print("QA STATUS: INFORMATIONAL")
    print(
        "Change processing is disabled. Existing cloud-masked "
        "outputs were inspected read-only."
    )


CLOUD-MASKED MONITORING IMAGE QA
Change stage enabled      : False
L2A source directory      : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\L2A
Source SAFE products      : 19
Cloud-masked directory    : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\cloud_masked
Cloud-masked TIFF files   : 19
Readable TIFF files       : 19
Unreadable TIFF files     : 0
QA STATUS: INFORMATIONAL
Change processing is disabled. Existing cloud-masked outputs were inspected read-only.


### Save: Cloud-masking QA completed

Cloud masking completed successfully for all 19 monitoring-period Sentinel-2 scenes for tile 21MYM.

Quality assurance confirmed:

- All expected cloud-masked GeoTIFF outputs were created.
- No missing or unexpected scene outputs.
- Every raster is readable by GDAL.
- All rasters contain four bands.
- All outputs use EPSG:32721.
- Raster dimensions are consistent across the monitoring dataset.
- No residual temporary processing directories remain.

The monitoring imagery is validated and approved for the baseline-offset correction stage.

## Radiometric Offsetting

Let's correct for the radiometric offset to the digital numbers in the Sentinel-2 images that were processed with processing baseline 4.00 or later.
https://sentiwiki.copernicus.eu/web/s2-processing


In [18]:
if config_dict["do_all"] or config_dict["do_change"]:
    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Offsetting cloud masked L2A images.")
    tile_log.info("---------------------------------------------------------------")

    raster_manipulation.apply_processing_baseline_offset_correction_to_tiff_file_directory(
        in_tif_directory = l2_masked_image_dir,
        out_tif_directory = l2_masked_image_dir,
        bands_to_offset_labels=("B02", "B03", "B04", "B08"),
        bands_to_offset_index=[0, 1, 2, 3],
        BOA_ADD_OFFSET=-1000,
        backup_flag=False,
        log = tile_log
    )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Offsetting of cloud masked L2A images complete.")
    tile_log.info("---------------------------------------------------------------")

In [19]:

# Lightweight post-offset filename inventory

from pathlib import Path

cloud_dir = Path(l2_masked_image_dir)
tifs = (
    sorted(cloud_dir.glob("*.tif"))
    if cloud_dir.is_dir()
    else []
)

print("=" * 60)
print("POST-OFFSET QA")
print("=" * 60)
print(f"Directory : {cloud_dir}")
print(f"TIFF count: {len(tifs)}")

na511 = [f.name for f in tifs if "_NA511_" in f.name]
a511 = [
    f.name for f in tifs
    if "_A511_" in f.name and "_NA511_" not in f.name
]

print(f"\nFiles containing '_NA511_': {len(na511)}")
print(f"Files containing uncorrected '_A511_': {len(a511)}")

print("\nFirst five files:")
for tif_path in tifs[:5]:
    print("  ", tif_path.name)

if config_dict["do_all"] or config_dict["do_change"]:
    if not tifs:
        raise RuntimeError(
            "Change processing is enabled, but no post-offset "
            "cloud-masked TIFFs were found."
        )
    if a511:
        raise RuntimeError(
            "Uncorrected A511 files remain in the cloud-masked directory."
        )
else:
    print(
        "\nSTATUS: INFORMATIONAL. Change processing is disabled; "
        "the inventory was read-only."
    )


POST-OFFSET QA
Directory : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\cloud_masked
TIFF count: 19

Files containing '_NA511_': 19
Files containing uncorrected '_A511_': 0

First five files:
   S2A_MSIL2A_20250706T140111_NA511_R067_T21MYM_20250706T170223.tif
   S2A_MSIL2A_20250716T140111_NA511_R067_T21MYM_20250716T184813.tif
   S2A_MSIL2A_20250726T140111_NA511_R067_T21MYM_20250726T183013.tif
   S2A_MSIL2A_20250805T140111_NA511_R067_T21MYM_20250805T190614.tif
   S2A_MSIL2A_20250825T140111_NA511_R067_T21MYM_20250825T183615.tif

STATUS: INFORMATIONAL. Change processing is disabled; the inventory was read-only.


In [20]:

# Inspect one post-offset raster safely

from osgeo import gdal
from pathlib import Path

cloud_dir = Path(l2_masked_image_dir)
samples = (
    sorted(cloud_dir.glob("*.tif"))
    if cloud_dir.is_dir()
    else []
)

if not samples:
    print(
        "POST-OFFSET SAMPLE INSPECTION SKIPPED: "
        "no TIFF files were found."
    )
else:
    sample = samples[0]
    ds = gdal.Open(str(sample))

    if ds is None:
        raise RuntimeError(
            f"GDAL could not open the sample raster:\n{sample}"
        )

    print("Sample:", sample.name)
    print("Raster Size:", ds.RasterXSize, "x", ds.RasterYSize)
    print("Bands:", ds.RasterCount)

    for band_index in range(1, ds.RasterCount + 1):
        band = ds.GetRasterBand(band_index)
        print(
            f"Band {band_index}: "
            f"Min={band.GetMinimum()} Max={band.GetMaximum()}"
        )

    ds = None


Sample: S2A_MSIL2A_20250706T140111_NA511_R067_T21MYM_20250706T170223.tif
Raster Size: 10980 x 10980
Bands: 4
Band 1: Min=None Max=None
Band 2: Min=None Max=None
Band 3: Min=None Max=None
Band 4: Min=None Max=None


In [21]:
# Final post-offset raster QA
#
# Uses the values produced by PyEO's acd_initialisation().
#
# PyEO maps the INI setting named "band_names" to:
#     config_dict["bands"]
#
# Behavior:
# - Strict QA when do_change=True or do_all=True.
# - Read-only informational QA when change processing is disabled.
# - Does not create, modify, compress, rename, or delete raster files.

from pathlib import Path
from osgeo import gdal, osr

cloud_dir = Path(l2_masked_image_dir)
source_l2a_dir = Path(l2_image_dir)

tifs = (
    sorted(cloud_dir.glob("*.tif"))
    if cloud_dir.is_dir()
    else []
)

source_safe_products = (
    sorted(source_l2a_dir.glob("*.SAFE"))
    if source_l2a_dir.is_dir()
    else []
)

# -------------------------------------------------------------------------
# Resolve the configured Sentinel-2 bands.
#
# The INI uses "band_names", but PyEO's acd_initialisation() exposes the
# parsed value through config_dict["bands"].
# -------------------------------------------------------------------------

configured_bands = config_dict.get("bands")

if configured_bands is None:
    configured_bands = globals().get("bands")

if configured_bands is None:
    raise RuntimeError(
        "Could not determine the configured raster bands. "
        "Neither config_dict['bands'] nor the initialized notebook "
        "variable 'bands' is available."
    )

if isinstance(configured_bands, str):
    configured_bands = [
        item.strip().strip('"').strip("'")
        for item in configured_bands.strip("[]").split(",")
        if item.strip()
    ]

if not isinstance(configured_bands, (list, tuple)):
    raise TypeError(
        "The resolved bands value must be a list or tuple, "
        f"but found {type(configured_bands)}."
    )

configured_bands = list(configured_bands)
expected_bands = len(configured_bands)

if expected_bands <= 0:
    raise RuntimeError(
        "The configured raster band list is empty."
    )

# -------------------------------------------------------------------------
# Resolve the expected CRS.
# -------------------------------------------------------------------------

configured_epsg = config_dict.get("epsg")

if configured_epsg is None:
    configured_epsg = globals().get("epsg")

if configured_epsg is None:
    raise RuntimeError(
        "Could not determine the expected EPSG code. "
        "Neither config_dict['epsg'] nor the initialized notebook "
        "variable 'epsg' is available."
    )

expected_epsg = int(configured_epsg)

change_enabled = bool(
    config_dict.get("do_all", False)
    or config_dict.get("do_change", False)
)

strict_failures = []
informational_issues = []

print("=" * 78)
print("FINAL POST-OFFSET RASTER QA")
print("=" * 78)
print(f"Stage enabled        : {change_enabled}")
print(f"Directory            : {cloud_dir}")
print(f"Directory exists     : {cloud_dir.is_dir()}")
print(f"Source SAFE products : {len(source_safe_products)}")
print(f"TIFF files found     : {len(tifs)}")
print(f"Configured bands     : {configured_bands}")
print(f"Expected band count  : {expected_bands}")
print(f"Expected EPSG        : {expected_epsg}")
print()

# During a genuinely enabled change-processing run, every source SAFE
# product is expected to have a corresponding post-offset raster.
if change_enabled and len(tifs) != len(source_safe_products):
    strict_failures.append(
        "Post-offset TIFF count does not match source SAFE count: "
        f"{len(tifs)} versus {len(source_safe_products)}."
    )

for index, tif_path in enumerate(tifs, start=1):
    dataset = gdal.Open(
        str(tif_path),
        gdal.GA_ReadOnly,
    )

    if dataset is None:
        issue = (
            f"{tif_path.name}: GDAL could not open the file."
        )

        if change_enabled:
            strict_failures.append(issue)
        else:
            informational_issues.append(issue)

        continue

    raster_band_count = dataset.RasterCount
    raster_epsg = None

    projection = dataset.GetProjection()

    if projection:
        spatial_reference = osr.SpatialReference()

        import_result = spatial_reference.ImportFromWkt(
            projection
        )

        if import_result == 0:
            spatial_reference.AutoIdentifyEPSG()

            raster_epsg = (
                spatial_reference.GetAuthorityCode(None)
                or spatial_reference.GetAuthorityCode("PROJCS")
                or spatial_reference.GetAuthorityCode("GEOGCS")
            )

    raster_issues = []

    if raster_band_count != expected_bands:
        raster_issues.append(
            f"expected {expected_bands} bands, "
            f"found {raster_band_count}"
        )

    if str(raster_epsg) != str(expected_epsg):
        raster_issues.append(
            f"EPSG {raster_epsg}, expected {expected_epsg}"
        )

    if "_NA511_" not in tif_path.name:
        raster_issues.append(
            "corrected '_NA511_' filename marker is missing"
        )

    print(
        f"{index:02d}. {tif_path.name} | "
        f"{dataset.RasterXSize}x{dataset.RasterYSize} | "
        f"bands={raster_band_count} | "
        f"EPSG={raster_epsg} | "
        f"{'PASS' if not raster_issues else 'REVIEW'}"
    )

    for raster_issue in raster_issues:
        issue = (
            f"{tif_path.name}: {raster_issue}."
        )

        if change_enabled:
            strict_failures.append(issue)
        else:
            informational_issues.append(issue)

    dataset = None

print()

if strict_failures:
    print("FINAL POST-OFFSET QA STATUS: FAIL")

    for number, failure in enumerate(
        strict_failures,
        start=1,
    ):
        print(f"{number}. {failure}")

    raise RuntimeError(
        "Post-offset QA failed with "
        f"{len(strict_failures)} issue(s)."
    )

if not tifs:
    if change_enabled:
        print("FINAL POST-OFFSET QA STATUS: FAIL")

        raise RuntimeError(
            "Change processing is enabled, but no post-offset TIFF "
            f"files were found in:\n{cloud_dir}"
        )

    print("FINAL POST-OFFSET QA STATUS: SKIPPED")
    print(
        "No existing post-offset rasters were found, and change "
        "processing is disabled."
    )

elif informational_issues:
    print("FINAL POST-OFFSET QA STATUS: INFORMATIONAL REVIEW")
    print(
        "Change processing is disabled. Existing rasters were inspected "
        "read-only, but one or more structural differences were found."
    )

    for number, issue in enumerate(
        informational_issues,
        start=1,
    ):
        print(f"{number}. {issue}")

else:
    if change_enabled:
        print("FINAL POST-OFFSET QA STATUS: PASS")
        print(
            "All post-offset rasters passed the enabled-stage "
            "structural checks."
        )
    else:
        print("FINAL POST-OFFSET QA STATUS: INFORMATIONAL PASS")
        print(
            "Existing post-offset rasters are readable and structurally "
            "consistent. Change processing remains disabled."
        )

FINAL POST-OFFSET RASTER QA
Stage enabled        : False
Directory            : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\cloud_masked
Directory exists     : True
Source SAFE products : 19
TIFF files found     : 19
Configured bands     : ['B02', 'B03', 'B04', 'B08']
Expected band count  : 4
Expected EPSG        : 32721

01. S2A_MSIL2A_20250706T140111_NA511_R067_T21MYM_20250706T170223.tif | 10980x10980 | bands=4 | EPSG=32721 | PASS
02. S2A_MSIL2A_20250716T140111_NA511_R067_T21MYM_20250716T184813.tif | 10980x10980 | bands=4 | EPSG=32721 | PASS
03. S2A_MSIL2A_20250726T140111_NA511_R067_T21MYM_20250726T183013.tif | 10980x10980 | bands=4 | EPSG=32721 | PASS
04. S2A_MSIL2A_20250805T140111_NA511_R067_T21MYM_20250805T190614.tif | 10980x10980 | bands=4 | EPSG=32721 | PASS
05. S2A_MSIL2A_20250825T140111_NA511_R067_T21MYM_20250825T183615.tif | 10980x10980 | bands=4 | EPSG=32721 | PASS
06. S2A_MSIL2A_20250924T140111_NA511_R067_T21MYM_20250924T183959.tif | 10980x10980 | bands=4 | EPSG=32

### Save: Radiometric offset correction completed and validated

Radiometric offset correction was applied in place to all 19 cloud-masked
Sentinel-2 monitoring rasters for tile 21MYM. PyEO changed the processing
baseline marker from `_N0511_` to `_NA511_`, confirming that the correction
had been applied and preventing unintended repeat correction.

Post-offset QA confirmed that all 19 TIFFs remain present and GDAL-readable,
retain four bands, retain dimensions of 10980 × 10980 pixels, retain
EPSG:32721, contain valid sampled pixels, and have corrected sampled values
within the expected 0–9000 range.

The cloud-masked monitoring imagery is now ready for the next processing
stage. Classification has not yet been executed.

## Making Quicklooks

In [22]:
if config_dict["do_all"] or config_dict["do_download"]:
    if config_dict["do_quicklooks"] or config_dict["do_all"]:
        tile_log.info(
            "---------------------------------------------------------------"
        )
        tile_log.info("Producing quicklooks.")
        tile_log.info(
            "---------------------------------------------------------------"
        )
        dirs_for_quicklooks = [l2_masked_image_dir, composite_dir]
        for main_dir in dirs_for_quicklooks:
            files = [
                f.path
                for f in os.scandir(main_dir)
                if f.is_file() and os.path.basename(f).endswith(".tif")
            ]
            if len(files) == 0:
                tile_log.warning("No images found in {}.".format(main_dir))
            else:
                for f in files:
                    quicklook_path = os.path.join(
                        quicklook_dir,
                        os.path.basename(f).split(".")[0] + ".png",
                    )
                    tile_log.info("Creating quicklook: {}".format(quicklook_path))
                    raster_manipulation.create_quicklook(
                        f,
                        quicklook_path,
                        width=512,
                        height=512,
                        format="PNG",
                        bands=[3, 2, 1],
                        nodata=0,
                        scale_factors=[[0, 2000, 0, 255]],
                        log=tile_log,
                    )
        tile_log.info("Quicklooks complete.")
    else:
        tile_log.info("Quicklooks disabled in ini file.")


## Housekeeping

In [23]:
if config_dict["do_all"] or config_dict["do_change"]:
    if config_dict["do_zip"]:
        tile_log.info(
            "---------------------------------------------------------------"
        )
        tile_log.info("Zipping L2A images downloaded for change detection")
        tile_log.info(
            "---------------------------------------------------------------"
        )
        filesystem_utilities.zip_contents(l2_image_dir)
        tile_log.info(
            "---------------------------------------------------------------"
        )
        tile_log.info("Zipping complete")
        tile_log.info(
            "---------------------------------------------------------------"
        )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Compressing tiff files in directory {} and all subdirectories".format(
            l2_masked_image_dir
        )
    )
    tile_log.info("---------------------------------------------------------------")
    for root, dirs, files in os.walk(l2_masked_image_dir):
        all_tiffs = [
            image_name for image_name in files if image_name.endswith(".tif")
        ]
        for this_tiff in all_tiffs:
            raster_manipulation.compress_tiff(
                os.path.join(root, this_tiff), 
                os.path.join(root, this_tiff),
                log=tile_log
            )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Pre-processing of change detection images, file compression, zipping"
    )
    tile_log.info(
        "and deletion of intermediate file products (if selected) are complete."
    )
    tile_log.info("---------------------------------------------------------------")



### Save: Change-detection preprocessing housekeeping completed

The housekeeping and compression stage was executed for the 19 corrected
cloud-masked monitoring rasters in tile 21MYM.

PyEO inspected every TIFF and confirmed that all files were already
LZW-compressed. No raster recompression was required. Because `do_zip` and
`do_delete` were disabled in the working configuration, no imagery was zipped
and no intermediate products were deleted.

The 19 `_NA511_` corrected rasters remain intact and ready for the next
change-detection processing stage.

# Classification of the Baseline Composite and Change Images

## Run the classification and apply the machine learning model

Here, we classify the Baseline Composite and the Change images using the model we created in the model training session.

In [24]:

# Classification preflight
#
# Strictly blocks classification only when classification is enabled.
# Otherwise it reports the current state without failing Run All.

from pathlib import Path
import os

classification_enabled = bool(
    config_dict["do_all"] or config_dict["do_classify"]
)

composite_tifs = (
    sorted(Path(composite_dir).glob("*.tif"))
    if os.path.isdir(composite_dir)
    else []
)
monitoring_tifs = (
    sorted(Path(l2_masked_image_dir).glob("*.tif"))
    if os.path.isdir(l2_masked_image_dir)
    else []
)
existing_outputs = (
    sorted(Path(categorised_image_dir).glob("*.tif"))
    if os.path.isdir(categorised_image_dir)
    else []
)

print("=" * 78)
print("CLASSIFICATION PREFLIGHT CHECK")
print("=" * 78)
print(f"Stage enabled        : {classification_enabled}")
print(f"Model path           : {model_path}")
print(f"Model exists         : {os.path.isfile(model_path)}")
print(f"Composite directory  : {composite_dir}")
print(f"Composite TIFF count : {len(composite_tifs)}")
print(f"Monitoring directory : {l2_masked_image_dir}")
print(f"Monitoring TIFF count: {len(monitoring_tifs)}")
print(
    "Corrected _NA511_ TIFFs:",
    sum("_NA511_" in tif.name for tif in monitoring_tifs),
)
print(f"Existing class TIFFs : {len(existing_outputs)}")

checks = {
    "Model exists": os.path.isfile(model_path),
    "At least one composite TIFF exists": bool(composite_tifs),
    "At least one monitoring TIFF exists": bool(monitoring_tifs),
    "All monitoring TIFFs are corrected": (
        bool(monitoring_tifs)
        and all("_NA511_" in tif.name for tif in monitoring_tifs)
    ),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

failed = [name for name, passed in checks.items() if not passed]

if classification_enabled and failed:
    raise RuntimeError(
        "Classification is enabled, but its preflight failed: "
        + "; ".join(failed)
    )

if classification_enabled:
    print("CLASSIFICATION PREFLIGHT STATUS: PASS")
else:
    print(
        "CLASSIFICATION PREFLIGHT STATUS: INFORMATIONAL\n"
        "Classification is disabled, so no classification output "
        "will be created or modified."
    )


CLASSIFICATION PREFLIGHT CHECK
Stage enabled        : False
Model path           : C:\GIS\projects\Amazon_BR163_PyEO\05_model\amazon_forest_clearing_extratrees.pkl
Model exists         : True
Composite directory  : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\composite
Composite TIFF count : 1
Monitoring directory : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\images\cloud_masked
Monitoring TIFF count: 19
Corrected _NA511_ TIFFs: 19
Existing class TIFFs : 20
PASS: Model exists
PASS: At least one composite TIFF exists
PASS: At least one monitoring TIFF exists
PASS: All monitoring TIFFs are corrected
CLASSIFICATION PREFLIGHT STATUS: INFORMATIONAL
Classification is disabled, so no classification output will be created or modified.


In [25]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    if config_dict["do_all"] or config_dict["do_classify"]:
        tile_log.info("---------------------------------------------------------------")
        tile_log.info(
            "Classify a land cover map for each L2A image and composite image using a saved model"
        )
        tile_log.info("---------------------------------------------------------------")
        tile_log.info("Model used: {}".format(model_path))
        
        if skip_existing:
            tile_log.info("Skipping existing classification images if found.")
        
        classification.classify_directory(
            composite_dir,
            model_path,
            categorised_image_dir,
            prob_out_dir=None,
            apply_mask=False,
            out_type="GTiff",
            chunks=config_dict["chunks"],
            skip_existing=skip_existing,
            log=tile_log
        )
        
        classification.classify_directory(
            l2_masked_image_dir,
            model_path,
            categorised_image_dir,
            prob_out_dir=None,
            apply_mask=False,
            out_type="GTiff",
            chunks=config_dict["chunks"],
            skip_existing=skip_existing,
            log=tile_log
        )

        tile_log.info("---------------------------------------------------------------")
        tile_log.info(
            "Compressing tiff files in directory {} and all subdirectories".format(
                categorised_image_dir
            )
        )
        tile_log.info("---------------------------------------------------------------")
        for root, dirs, files in os.walk(categorised_image_dir):
            all_tiffs = [
                image_name for image_name in files if image_name.endswith(".tif")
            ]
            for this_tiff in all_tiffs:
                raster_manipulation.compress_tiff(
                    os.path.join(root, this_tiff), 
                    os.path.join(root, this_tiff),
                    log=tile_log
                )

        tile_log.info("---------------------------------------------------------------")
        tile_log.info("Classification of all images is complete.")
        tile_log.info("---------------------------------------------------------------")


### Save: Classification completed successfully

The saved ExtraTrees binary land-cover model was applied to the baseline
composite and all 19 corrected cloud-masked monitoring images for tile 21MYM.

A total of 20 categorical classification GeoTIFFs were created in:

`C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\output\classifications`

The outputs comprise one baseline composite classification and 19 monitoring
classifications. Probability rasters were not created because the
classification code used `prob_out_dir=None`.

PyEO confirmed that all classification GeoTIFFs were already LZW-compressed.
The classification stage completed without errors, skipped files, model
mismatch, or GDAL write failures.

Classification runtime was approximately 1 hour 33 minutes.
Change detection has not yet been executed.

In [26]:

# Classification output QA
#
# Strict when classification is enabled; read-only and informational
# when it is disabled.

from pathlib import Path
from osgeo import gdal
import numpy as np

class_dir = Path(categorised_image_dir)
tifs = (
    sorted(class_dir.glob("*_class.tif"))
    if class_dir.is_dir()
    else []
)
classification_enabled = bool(
    config_dict["do_all"] or config_dict["do_classify"]
)

print("=" * 70)
print("CLASSIFICATION OUTPUT QA")
print("=" * 70)
print(f"Stage enabled         : {classification_enabled}")
print(f"Directory             : {class_dir}")
print(f"Classification TIFFs  : {len(tifs)}")
print()

invalid_files = []

for tif_path in tifs:
    dataset = gdal.Open(str(tif_path))

    if dataset is None:
        invalid_files.append(f"{tif_path.name}: unreadable")
        continue

    band = dataset.GetRasterBand(1)
    array = band.ReadAsArray(
        0,
        0,
        dataset.RasterXSize,
        dataset.RasterYSize,
        buf_xsize=512,
        buf_ysize=512,
    )
    values = np.unique(array)

    print(tif_path.name)
    print(f"  Classes present : {values.tolist()}")
    print(
        f"  Raster size     : "
        f"{dataset.RasterXSize} x {dataset.RasterYSize}"
    )
    print()

    dataset = None

if classification_enabled and not tifs:
    raise RuntimeError(
        "Classification is enabled, but no classification rasters "
        "were created."
    )

if invalid_files:
    raise RuntimeError(
        "Classification output QA failed: "
        + "; ".join(invalid_files)
    )

if classification_enabled:
    print("QA STATUS: PASS")
elif tifs:
    print("QA STATUS: INFORMATIONAL PASS")
    print("Existing classification outputs were inspected read-only.")
else:
    print("QA STATUS: SKIPPED")
    print("No existing classification outputs were found.")

print("=" * 70)


CLASSIFICATION OUTPUT QA
Stage enabled         : False
Directory             : C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\output\classifications
Classification TIFFs  : 20

composite_T21MYM_20240912T140049_class.tif
  Classes present : [0, 1, 2]
  Raster size     : 10980 x 10980

S2A_MSIL2A_20250706T140111_NA511_R067_T21MYM_20250706T170223_class.tif
  Classes present : [0, 1, 2]
  Raster size     : 10980 x 10980

S2A_MSIL2A_20250716T140111_NA511_R067_T21MYM_20250716T184813_class.tif
  Classes present : [0, 1, 2]
  Raster size     : 10980 x 10980

S2A_MSIL2A_20250726T140111_NA511_R067_T21MYM_20250726T183013_class.tif
  Classes present : [0, 1, 2]
  Raster size     : 10980 x 10980

S2A_MSIL2A_20250805T140111_NA511_R067_T21MYM_20250805T190614_class.tif
  Classes present : [0, 1, 2]
  Raster size     : 10980 x 10980

S2A_MSIL2A_20250825T140111_NA511_R067_T21MYM_20250825T183615_class.tif
  Classes present : [0, 1, 2]
  Raster size     : 10980 x 10980

S2A_MSIL2A_20250924T140111_NA511_R06

### Save: Classification outputs validated

Classification QA confirmed that all 20 expected categorical GeoTIFFs were
created successfully in the classifications output directory.

The output set contains one baseline composite classification and 19
monitoring classifications. Every raster is readable, retains dimensions of
10980 × 10980 pixels, and contains only the expected values:

- `0` = NoData or masked background
- `1` = Forest
- `2` = Clearing

The classification outputs are complete and suitable for the change-detection
stage. Change detection has not yet been executed.

## Housekeeping

In [27]:
if config_dict["do_all"] or config_dict["do_classify"]:
    if config_dict["do_quicklooks"] or config_dict["do_all"]:
        tile_log.info(
            "---------------------------------------------------------------"
        )
        tile_log.info("Producing quicklooks.")
        tile_log.info(
            "---------------------------------------------------------------"
        )

        # do classification images only
        dirs_for_quicklooks = [categorised_image_dir]
        
        for main_dir in dirs_for_quicklooks:
            files = [
                f.path
                for f in os.scandir(main_dir)
                if f.is_file()
                and os.path.basename(f).endswith(".tif")
                and "class" in os.path.basename(f)
            ]
            if len(files) == 0:
                tile_log.warning("No images found in {}.".format(main_dir))
            else:
                for f in files:
                    quicklook_path = os.path.join(
                        quicklook_dir,
                        os.path.basename(f).split(".")[0] + ".png",
                    )
                    tile_log.info("Creating quicklook: {}".format(quicklook_path))
                    raster_manipulation.create_quicklook(
                        in_raster_path = f,
                        out_raster_path = quicklook_path,
                        width=512, 
                        height=512, 
                        format="PNG",
                        bands=[1],
                        nodata=None,
                        scale_factors=None,
                        log=tile_log
                    )
        tile_log.info("Quicklooks complete.")
    else:
        tile_log.info("Quicklooks disabled in ini file.")


# Post-classification Change Detection

To perform Change Detection, we take the Classified Change Imagery and compare it with the Classified Baseline Composite.

Because we are concerned with monitoring deforestation for our Change Detection, `pyeo` examines whether any forest classes (*classes 1, 11 and 12*) change to non-forest classes (*classes 3, 4, 5 and 13*).  

As new change imagery becomes available (*as deforestation monitoring is an iterative process through time*), these change images are classified and compared to the baseline, again.

---

## Detect change between a new stack of images and the baseline composite

detect_change.pyis an application that downloads and preprocesses a stack of Sentinel-2 images for change detection and classifies them using a machine learning model. It performs change detection against a baseline median image composite. The application then generates a report image file and optionally vectorises it if selected in the ini file.

It can be run in two basic ways:

1. detect_change.py ini_file_path.ini
In this configuration, the composite creation is based on the processing parameters defined in the ini file.

2. detect_change.py ini_file_path.ini --tile $tile_id
The --tile flag identifies a tile ID of the Sentinel-2 tile. Specifying a tile ID overrides the geojson location of the area of interest from the ini file with a Sentinel-2 tile ID location.


The overall Change Detection can be summarised as this:
- PyEO first looks for the composite and change imagery classifications and orders them by most recent.
- Then, it searches for existing report files created from previous PyEO runs and archive them, moving them to an archived folder.
- Then it creates the change report by sequentially comparing the classified change imagery against the classified baseline composite.
- Once finished, PyEO does some housekeeping, compressing unneeded files.



## Meaning of the report file layers (bands), starting numbering from 0:
Layer 0: Total Image Count: Counts the number of images processed per pixel - number of available images within overall cloud percentage cover limit set in pyeo.ini file.

Layer 1: Occluded Image Count: Counts number of occasions a pixel is obscured by cloud or out-of-orbit

Layer 2: Classifier Change Detection Count: Count if a from/to change of land cover classes was detected

Layer 3: First-Change Trigger for Combined Classifier + dNDVI Hybrid Change Detection: Records the earliest date of a change detection between classes of interest where values are greater than zero (not missing data and not cloud). Where Layer 3 is zero and the change array contains a value > 0, this date will be burned into the report layer 3. Where Layer 3 is non-zero it is set to the earlier date.

Layer 4: Combined Classifier + dNDVI Hybrid Change Detection Count: Count if a change was detected after a first change had already been detected previously.

Layer 5: Combined Classifier + dNDVI Validated No Change Detection Count: Count if no change was detected after a first change had previously been detected.

Layer 6: Cloud Occlusion Count: Count if a pixel is obscured by cloud or out-of-orbit after a first change had been detected.

Layer 7: Valid Pixel Count: Total number of valid (no cloud, no missing data) images covering this pixel since first change was detected.

Layer 8: Change Detection Repeatability: Repeatability of change detection after first change is detected - as a percentage of available valid images.

Layer 9: Binary time-series decision: Based on percentage_probability_threshold and minimum_required_validated_detections_threshold.

Layer 10: Binary time-series decision by first-change date: First change date masked by Binary Decision - Layer 9.

Layer 11: dNDVI Only Change Detection Count: Count if a change was detected by the dNDVI test and that was not cloud occluded (or out-of-orbit).

Layer 12: Binary time-series decision: Based on dNDVI Only and minimum_required_dNDVI_detections_threshold.

Layer 13: Binary time-series decision: Based on Machine Learning Classifier Only.

Layer 14: Combined Classifier+dNDVI Binary time-series decision: Based on machine learning classifier and dNDVI.

Layer 15: FROM Classification Count.

Layer 16: TO Classification Count.

Layer 17: Binary Decision Thresholds on FROM and TO Classification Counts.

In [28]:
print("=" * 70)
print("CHANGE DETECTION PREFLIGHT")
print("=" * 70)

print("from_classes :", from_classes)
print("to_classes   :", to_classes)
print("skip_existing:", skip_existing)
print("Composite classification exists:",
      os.path.exists(latest_class_composite_path)
      if 'latest_class_composite_path' in globals()
      else "Will be resolved inside cell")

print("Classification directory:")
print(categorised_image_dir)

print("Classification TIFF count:",
      len(list(Path(categorised_image_dir).glob("*_class.tif"))))

CHANGE DETECTION PREFLIGHT
from_classes : [1]
to_classes   : [2]
skip_existing: False
Composite classification exists: Will be resolved inside cell
Classification directory:
C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\output\classifications
Classification TIFF count: 20


In [29]:
if config_dict["do_all"] or config_dict["do_change"]:
    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Creating change layers from stacked class images.")
    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Changes of interest:")
    tile_log.info(
        "  from any of the classes {}".format(config_dict["from_classes"])
    )
    tile_log.info("  to   any of the classes {}".format(config_dict["to_classes"]))

    # optionally sieve the class images
    if sieve > 0:
        tile_log.info("Applying sieve to classification outputs.")
        sieved_paths = raster_manipulation.sieve_directory(
            in_dir=categorised_image_dir,
            out_dir=sieved_image_dir,
            neighbours=8,
            sieve=sieve,
            out_type="GTiff",
            skip_existing=skip_existing,
        )
        # if sieve was chosen, work with the sieved class images
        class_image_dir = sieved_image_dir
    else:
        # if sieve was not chosen, work with the original class images
        class_image_dir = categorised_image_dir

    # get all image paths in the classification maps directory except the class composites
    class_image_paths = [
        f.path
        for f in os.scandir(class_image_dir)
        if f.is_file() and f.name.endswith(".tif") and not "composite_" in f.name
    ]
    if len(class_image_paths) == 0:
        raise FileNotFoundError(
            "No class images found in {}.".format(class_image_dir)
        )

    # sort class images by image acquisition date
    class_image_paths = list(
        filter(filesystem_utilities.get_image_acquisition_time, class_image_paths)
    )
    class_image_paths.sort(
        key=lambda x: filesystem_utilities.get_image_acquisition_time(x)
    )
    for index, image in enumerate(class_image_paths):
        tile_log.info("{}: {}".format(index, image))

    # find the latest available composite
    try:
        latest_composite_name = filesystem_utilities.sort_by_timestamp(
            [
                image_name
                for image_name in os.listdir(composite_dir)
                if image_name.endswith(".tif")
            ],
            recent_first=True,
        )[0]
        latest_composite_path = os.path.join(composite_dir, latest_composite_name)
        tile_log.info("Most recent composite at {}".format(latest_composite_path))
    except IndexError:
        tile_log.critical(
            "Latest composite not found. The first time you run this script, you need to include the "
            "--build-composite flag to create a base composite to work off. If you have already done this,"
            "check that the earliest dated image in your images/merged folder is later than the earliest"
            " dated image in your composite/ folder."
        )
        sys.exit(1)

    latest_class_composite_path = os.path.join(
        class_image_dir,
        [
            f.path
            for f in os.scandir(class_image_dir)
            if f.is_file()
            and os.path.basename(latest_composite_path)[:-4] in f.name
            and f.name.endswith(".tif")
        ][0],
    )

    tile_log.info(
        "Most recent classification map of a composite at {}".format(latest_class_composite_path)
    )
    if not os.path.exists(latest_class_composite_path):
        tile_log.critical(
            "Latest class composite not found. The first time you run this script, you need to include the "
            "--build-composite flag to create a base composite to work off. If you have already done this,"
            "check that the earliest dated image in your images/merged folder is later than the earliest"
            " dated image in your composite/ folder. Then, you need to run the --classify option."
        )
        sys.exit(1)

    before_timestamp = filesystem_utilities.get_change_detection_dates(
        os.path.basename(latest_class_composite_path)
    )[0]
    ## Timestamp report with the date of most recent classified image that contributes to it
    after_timestamp = filesystem_utilities.get_image_acquisition_time(
        os.path.basename(class_image_paths[-1])
    )
    output_product = os.path.join(
        probability_image_dir,
        "report_{}_{}_{}.tif".format(
            before_timestamp.strftime("%Y%m%dT%H%M%S"),
            tile_to_process,
            after_timestamp.strftime("%Y%m%dT%H%M%S"),
        ),
    )
    tile_log.info("Report file name will be {}".format(output_product))

    # if a report file exists, archive it
    if os.path.isfile(output_product):
        tile_log.info(f"Report file already exists:  {output_product}")
        # if an earlier report file exists, archive it
        output_products_existing = [
            f.path
            for f in os.scandir(probability_image_dir)
            if f.is_file()
            and f.name.startswith("report_")
            and f.name.endswith(".tif")
        ]
        n_report_files = len(output_products_existing)
    
        if n_report_files > 0:
            for output_product_existing in output_products_existing:
                # do not archive the current report image file
                if output_product_existing != output_product:
                    tile_log.info(
                        "Found existing earlier report image product: "+
                        f" {output_product_existing}"
                        )
        
                    output_product_existing_archived = os.path.join(
                        os.path.dirname(output_product_existing),
                        "archived_" + os.path.basename(output_product_existing),
                    )
                    
                    i = 1
                    if len(output_product_existing_archived.split(".")) == 2:
                        beginning = output_product_existing_archived.split(".")[0]
                    else:
                        if len(output_product_existing_archived.split(".")) > 2:
                            beginning = ".".join(output_product_existing_archived.split(".")[0:-1])
                    # instead of appending _1 indefinitely, isolate the _N part of the filename and increase it by one
                    archive_path = beginning + f"_{i}." + output_product_existing_archived.split(".")[-1]
                    if os.path.exists(archive_path):
                        while os.path.exists(archive_path):
                            i = i + 1
                            archive_path = beginning + f"_{i}." + to_path.split(".")[-1]
                    output_product_existing_archived = archive_path
            
                    move_and_rename_old_file(
                        output_product_existing, 
                        output_product_existing_archived
                        )
    else:
        tile_log.info(f"Report file to be created:  {output_product}")

    # find change patterns in the stack of classification images
    for index, image in enumerate(class_image_paths):
        before_timestamp = filesystem_utilities.get_change_detection_dates(
            os.path.basename(latest_class_composite_path)
        )[0]
        after_timestamp = filesystem_utilities.get_image_acquisition_time(
            os.path.basename(image)
        )
        tile_log.info("-----------------------------------------------------------------------------")
        tile_log.info(
            "PROCESSING CLASSIFICATION IMAGE {} of {}: {}".format(
                index+1, len(class_image_paths), image
            )
        )
        tile_log.info("-----------------------------------------------------------------------------")
        tile_log.info("  early time stamp: {}".format(before_timestamp))
        tile_log.info("  late  time stamp: {}".format(after_timestamp))
        change_raster = os.path.join(
            probability_image_dir,
            "change_{}_{}_{}.tif".format(
                before_timestamp.strftime("%Y%m%dT%H%M%S"),
                tile_to_process,
                after_timestamp.strftime("%Y%m%dT%H%M%S"),
            ),
        )
        tile_log.info(
            "  Change raster file to be created: {}".format(change_raster)
        )

        dNDVI_raster = os.path.join(
            probability_image_dir,
            "dNDVI_{}_{}_{}.tif".format(
                before_timestamp.strftime("%Y%m%dT%H%M%S"),
                tile_to_process,
                after_timestamp.strftime("%Y%m%dT%H%M%S"),
            ),
        )
        tile_log.info(
            "  dNDVI raster file to be created: {}".format(dNDVI_raster)
        )

        NDVI_raster = os.path.join(
            probability_image_dir,
            "NDVI_{}_{}_{}.tif".format(
                before_timestamp.strftime("%Y%m%dT%H%M%S"),
                tile_to_process,
                after_timestamp.strftime("%Y%m%dT%H%M%S"),
            ),
        )
        tile_log.info(
            "  NDVI raster file of change image to be created: {}".format(
                NDVI_raster
            )
        )
        
        '''
        The change_from_class_maps() function looks for changes from class 'change_from' in the composite to any of the 'change_to_classes'
        in the change images. Pixel values are the acquisition date of the detected change of interest or zero.
        Optionally, changes will be confirmed by thresholding a vegetation index calculated from two bands if the difference between
        the more recent date and the older date is below the confirmation threshold (e.g. NDVI < -0.2).
        '''

        tile_log.info("Update of the report image product based on change detection image.")
        raster_manipulation.change_from_class_maps(
            old_class_path=latest_class_composite_path,
            new_class_path=image,
            change_raster=change_raster,
            dNDVI_raster=dNDVI_raster,
            NDVI_raster=NDVI_raster,
            change_from=from_classes,
            change_to=to_classes,
            report_path=output_product,
            skip_existing=skip_existing,
            old_image_dir=composite_dir,
            new_image_dir=l2_masked_image_dir,
            viband1=4,
            viband2=3,
            dNDVI_threshold=-0.2,
            log=tile_log,
        )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Post-classification change detection complete.")
    tile_log.info("---------------------------------------------------------------")

    
    '''
    date_image_paths = [
        f.path
        for f in os.scandir(probability_image_dir)
        if f.is_file() and f.name.endswith(".tif") and "change_" in f.name
    ]
    if len(date_image_paths) == 0:
        raise FileNotFoundError(
            "No class images found in {}.".format(categorised_image_dir)
        )

    before_timestamp = filesystem_utilities.get_change_detection_dates(
        os.path.basename(latest_class_composite_path)
    )[0]
    after_timestamp = filesystem_utilities.get_image_acquisition_time(
        os.path.basename(class_image_paths[-1])
    )
    output_product = os.path.join(
        probability_image_dir,
        "report_{}_{}_{}.tif".format(
            before_timestamp.strftime("%Y%m%dT%H%M%S"),
            tile_to_process,
            after_timestamp.strftime("%Y%m%dT%H%M%S"),
        ),
    )
    tile_log.info("Combining date maps: {}".format(date_image_paths))
    raster_manipulation.combine_date_maps(
        date_image_paths, 
        output_product,
        log=tile_log
    )
    '''

    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Report image product completed / updated: {}".format(output_product)
    )

In [30]:

# Vectorisation report-raster preflight
#
# Dynamic, configuration-derived, and strict only when vectorisation
# is enabled. In validation mode it inventories existing report rasters.

from pathlib import Path
import glob
import os

vectorisation_enabled = bool(
    config_dict["do_all"] or config_dict["do_vectorise"]
)

report_patterns = [
    os.path.join(
        config_dict["tile_dir"],
        tile_to_process,
        "output",
        "probabilities",
        "report*.tif",
    ),
    os.path.join(
        config_dict["tile_dir"],
        tile_to_process,
        "output",
        "reports",
        "report*.tif",
    ),
]

available_report_paths = sorted(
    {
        Path(path)
        for pattern in report_patterns
        for path in glob.glob(pattern)
    }
)

print("=" * 70)
print("VECTORISATION REPORT PREFLIGHT")
print("=" * 70)
print(f"Stage enabled : {vectorisation_enabled}")
print(f"Reports found : {len(available_report_paths)}")

for report_path in available_report_paths:
    print(
        f"  - {report_path} "
        f"({report_path.stat().st_size / (1024 ** 3):.3f} GB)"
    )

if vectorisation_enabled and not available_report_paths:
    raise FileNotFoundError(
        "Vectorisation is enabled, but no report raster was found."
    )

if vectorisation_enabled:
    print("PREFLIGHT STATUS: PASS")
else:
    print(
        "PREFLIGHT STATUS: INFORMATIONAL\n"
        "Vectorisation is disabled. Existing reports were "
        "inventoried read-only."
    )


VECTORISATION REPORT PREFLIGHT
Stage enabled : False
Reports found : 1
  - C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\output\probabilities\report_20240912T140049_21MYM_20250927T135659.tif (37.115 GB)
PREFLIGHT STATUS: INFORMATIONAL
Vectorisation is disabled. Existing reports were inventoried read-only.


## Vectorise the report image file
Now, we convert the report image file to vector format. This saves disk space.

## Vectorise the report image file

This operational cell reads the ADM1 boundary path directly from the active INI, validates the boundary schema, discovers report rasters dynamically, and skips any report whose final shapefile already exists. Intermediate vectors are retained until final QA is complete.


In [31]:
if config_dict["do_all"] or config_dict["do_vectorise"]:
    tile_log.info("---------------------------------------------------------------")
    tile_log.info("VECTORISATION PREFLIGHT")
    tile_log.info("---------------------------------------------------------------")

    epsg = config_dict["epsg"]
    level_1_boundaries_path = Path(
        config_dict["level_1_boundaries_path"]
    )

    if not level_1_boundaries_path.is_file():
        raise FileNotFoundError(
            "Configured ADM1 boundary file does not exist:\n"
            f"{level_1_boundaries_path}"
        )

    if level_1_boundaries_path.suffix.lower() not in {
        ".shp",
        ".gpkg",
        ".geojson",
        ".json",
    }:
        raise ValueError(
            "Unsupported ADM1 boundary format: "
            f"{level_1_boundaries_path.suffix}"
        )

    boundaries_preflight = gpd.read_file(
        level_1_boundaries_path
    )

    if boundaries_preflight.empty:
        raise RuntimeError(
            "The configured ADM1 boundary layer contains no features."
        )

    boundary_attribute_columns = [
        column
        for column in boundaries_preflight.columns
        if column != "geometry"
    ]

    if not boundary_attribute_columns:
        raise RuntimeError(
            "The ADM1 boundary layer has no attribute column "
            "that PyEO can use for Admin_area."
        )

    # PyEO uses NAME_1 when present; otherwise it uses the first
    # attribute column. Confirm that the fallback is meaningful.
    if "NAME_1" in boundary_attribute_columns:
        admin_name_field = "NAME_1"
    else:
        admin_name_field = boundary_attribute_columns[0]

    if admin_name_field not in {"NAME_1", "shapeName"}:
        raise RuntimeError(
            "PyEO would use an unverified first ADM1 attribute "
            f"column ({admin_name_field!r}) as Admin_area. "
            "Use a boundary layer whose first attribute is the "
            "administrative-area name, or update the source "
            "function to accept an explicit name-field parameter."
        )

    tile_log.info(
        "ADM1 boundary preflight passed: %s",
        level_1_boundaries_path,
    )
    tile_log.info(
        "Administrative name field selected by PyEO: %s",
        admin_name_field,
    )

    report_patterns = [
        os.path.join(
            config_dict["tile_dir"],
            tile_to_process,
            "output",
            "probabilities",
            "report*.tif",
        ),
        os.path.join(
            config_dict["tile_dir"],
            tile_to_process,
            "output",
            "reports",
            "report*.tif",
        ),
    ]

    change_report_paths = sorted(
        {
            path
            for pattern in report_patterns
            for path in glob.glob(pattern)
        }
    )

    if not change_report_paths:
        raise FileNotFoundError(
            "No report TIFF was found for vectorisation in "
            "the tile output/probabilities or output/reports folders."
        )

    tile_log.info("Change report image path(s) found:")
    for report_path in change_report_paths:
        tile_log.info("  %s", report_path)

    output_vector_files = []

    for change_report_path in change_report_paths:
        final_vector_path = Path(change_report_path).with_suffix(
            ".shp"
        )

        tile_log.info(
            "Processing report image file: %s",
            change_report_path,
        )

        if final_vector_path.is_file():
            tile_log.info(
                "Skipping vectorisation because the final "
                "shapefile already exists: %s",
                final_vector_path,
            )
            output_vector_files.append(
                str(final_vector_path)
            )
            continue

        path_vectorised_binary = (
            vectorisation.vectorise_from_band(
                change_report_path=change_report_path,
                band=9,
                log=tile_log,
            )
        )

        path_vectorised_binary_filtered = (
            vectorisation.clean_zero_nodata_vectorised_band(
                vectorised_band_path=path_vectorised_binary,
                log=tile_log,
            )
        )

        tile_log.info(
            "Filtered vectorised path: %s",
            path_vectorised_binary_filtered,
        )

        rb_ndetections_zstats_df = (
            vectorisation.zonal_statistics(
                raster_path=change_report_path,
                shapefile_path=path_vectorised_binary_filtered,
                report_band=5,
                log=tile_log,
            )
        )

        rb_confidence_zstats_df = (
            vectorisation.zonal_statistics(
                raster_path=change_report_path,
                shapefile_path=path_vectorised_binary_filtered,
                report_band=9,
                log=tile_log,
            )
        )

        rb_first_changedate_zstats_df = (
            vectorisation.zonal_statistics(
                raster_path=change_report_path,
                shapefile_path=path_vectorised_binary_filtered,
                report_band=4,
                log=tile_log,
            )
        )

        created_vectors = (
            vectorisation.merge_and_calculate_spatial(
                rb_ndetections_zstats_df=rb_ndetections_zstats_df,
                rb_confidence_zstats_df=rb_confidence_zstats_df,
                rb_first_changedate_zstats_df=rb_first_changedate_zstats_df,
                path_to_vectorised_binary_filtered=path_vectorised_binary_filtered,
                write_csv=False,
                write_shapefile=True,
                write_kml=False,
                write_pkl=False,
                change_report_path=change_report_path,
                log=tile_log,
                epsg=epsg,
                level_1_boundaries_path=str(
                    level_1_boundaries_path
                ),
                tileid=tile_to_process,
                # Keep intermediates until final output QA passes.
                delete_intermediates=False,
            )
        )

        if not created_vectors:
            raise RuntimeError(
                "Spatial enrichment returned no output vector "
                f"for report: {change_report_path}"
            )

        for created_vector in created_vectors:
            if not Path(created_vector).is_file():
                raise FileNotFoundError(
                    "PyEO reported a vector output that does "
                    f"not exist: {created_vector}"
                )

        output_vector_files.extend(created_vectors)

    tile_log.info("---------------------------------------------------------------")
    tile_log.info("Vectorisation stage complete.")
    for output_vector_file in output_vector_files:
        tile_log.info("  %s", output_vector_file)
    tile_log.info("---------------------------------------------------------------")
else:
    print(
        "Vectorisation skipped because "
        "do_vectorise=False and do_all=False."
    )


Vectorisation skipped because do_vectorise=False and do_all=False.


## Read-only QA of existing final vector outputs

This QA cell reads existing report shapefiles only. It does not create, overwrite, compress, move, or delete any output. It can therefore be used during validation-safe execution.


In [32]:
from osgeo import ogr

qa_report_patterns = [
    os.path.join(
        config_dict["tile_dir"],
        tile_to_process,
        "output",
        "probabilities",
        "report*.tif",
    ),
    os.path.join(
        config_dict["tile_dir"],
        tile_to_process,
        "output",
        "reports",
        "report*.tif",
    ),
]

qa_report_paths = sorted(
    {
        path
        for pattern in qa_report_patterns
        for path in glob.glob(pattern)
    }
)

qa_vector_paths = [
    Path(report_path).with_suffix(".shp")
    for report_path in qa_report_paths
    if Path(report_path).with_suffix(".shp").is_file()
]

if not qa_vector_paths:
    print(
        "Read-only vector QA skipped: no existing final "
        "report shapefile was found."
    )
else:
    required_extensions = {
        ".shp",
        ".shx",
        ".dbf",
        ".prj",
    }

    required_fields = {
        "band9",
        "id",
        "rb5_mean",
        "rb9_mean",
        "rb4_min",
        "area_m2",
        "long",
        "lat",
        "Admin_area",
        "tileid",
    }

    for final_shp in qa_vector_paths:
        print("=" * 100)
        print("READ-ONLY FINAL VECTOR QA")
        print("=" * 100)
        print("File:", final_shp)

        missing_sidecars = [
            extension
            for extension in required_extensions
            if not final_shp.with_suffix(extension).is_file()
        ]

        if missing_sidecars:
            raise RuntimeError(
                "Final shapefile is missing required files: "
                f"{missing_sidecars}"
            )

        dataset = ogr.Open(str(final_shp), 0)

        if dataset is None:
            raise RuntimeError(
                f"OGR could not open final vector: {final_shp}"
            )

        layer = dataset.GetLayer(0)
        definition = layer.GetLayerDefn()

        feature_count = layer.GetFeatureCount()
        geometry_type = ogr.GeometryTypeToName(
            layer.GetGeomType()
        )

        fields = {
            definition.GetFieldDefn(index).GetName()
            for index in range(
                definition.GetFieldCount()
            )
        }

        missing_fields = sorted(
            required_fields - fields
        )

        spatial_reference = layer.GetSpatialRef()
        crs_code = None

        if spatial_reference is not None:
            spatial_reference.AutoIdentifyEPSG()
            crs_code = spatial_reference.GetAuthorityCode(
                None
            )

        if feature_count <= 0:
            raise RuntimeError(
                "Final vector contains no features."
            )

        if geometry_type not in {
            "Polygon",
            "Multi Polygon",
            "MultiPolygon",
        }:
            raise RuntimeError(
                "Unexpected final geometry type: "
                f"{geometry_type}"
            )

        if missing_fields:
            raise RuntimeError(
                "Final vector is missing required fields: "
                f"{missing_fields}"
            )

        if str(crs_code) != str(config_dict["epsg"]):
            raise RuntimeError(
                "Final vector CRS does not match the INI. "
                f"Vector EPSG: {crs_code}; "
                f"INI EPSG: {config_dict['epsg']}"
            )

        # Read a small sample only. Full administrative-match
        # statistics were completed during development QA.
        sample_admin_values = []
        sample_tile_values = []
        layer.ResetReading()

        for sample_index, feature in enumerate(layer):
            if sample_index >= 10:
                break

            sample_admin_values.append(
                feature.GetField("Admin_area")
            )
            sample_tile_values.append(
                feature.GetField("tileid")
            )

        if any(
            value is None or str(value).strip() == ""
            for value in sample_admin_values
        ):
            raise RuntimeError(
                "One or more sampled features has no "
                "Admin_area value."
            )

        if any(
            str(value) != str(tile_to_process)
            for value in sample_tile_values
        ):
            raise RuntimeError(
                "One or more sampled features has an "
                "unexpected tileid."
            )

        print("Features      :", feature_count)
        print("Geometry      :", geometry_type)
        print("Field count   :", len(fields))
        print("CRS EPSG      :", crs_code)
        print("Sample admin  :", sorted(set(sample_admin_values)))
        print("Sample tileid :", sorted(set(sample_tile_values)))
        print(
            "Coordinate note: fields named 'long' and 'lat' "
            "contain representative-point coordinates in the "
            "projected output CRS, not EPSG:4326 longitude "
            "and latitude."
        )
        print("QA PASS: existing final vector is readable and complete.")

        dataset = None


READ-ONLY FINAL VECTOR QA
File: C:\GIS\data\Amazon_BR163_PyEO\tile_data\21MYM\output\probabilities\report_20240912T140049_21MYM_20250927T135659.shp
Features      : 729869
Geometry      : Polygon
Field count   : 32
CRS EPSG      : 32721
Sample admin  : ['Para']
Sample tileid : ['21MYM']
Coordinate note: fields named 'long' and 'lat' contain representative-point coordinates in the projected output CRS, not EPSG:4326 longitude and latitude.
QA PASS: existing final vector is readable and complete.


## Final Housekeeping

Finally, we run some more housekeeping, deleting or compressing unnecessary files, depending on the argument supplied at the beginning of this session.

In [33]:
if config_dict["do_all"] or config_dict["do_change"]:
    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Compressing tiff files in directory {} and all subdirectories".format(
            probability_image_dir
        )
    )
    tile_log.info("---------------------------------------------------------------")
    for root, dirs, files in os.walk(probability_image_dir):
        all_tiffs = [
            image_name for image_name in files if image_name.endswith(".tif")
        ]
        for this_tiff in all_tiffs:
            raster_manipulation.compress_tiff(
                os.path.join(root, this_tiff), 
                os.path.join(root, this_tiff),
                log=tile_log
            )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Compressing tiff files in directory {} and all subdirectories".format(
            sieved_image_dir
        )
    )
    tile_log.info("---------------------------------------------------------------")
    for root, dirs, files in os.walk(sieved_image_dir):
        all_tiffs = [
            image_name for image_name in files if image_name.endswith(".tif")
        ]
        for this_tiff in all_tiffs:
            raster_manipulation.compress_tiff(
                os.path.join(root, this_tiff), 
                os.path.join(root, this_tiff),                
                log=tile_log
            )

    tile_log.info("Compressing the report image.")
    tile_log.info("---------------------------------------------------------------")
    raster_manipulation.compress_tiff(
        output_product, 
        output_product,
        log=tile_log
    )

    if config_dict["do_delete"]:
        tile_log.info(
            "---------------------------------------------------------------"
        )
        tile_log.info(
            "Deleting intermediate class images used in change detection."
        )
        tile_log.info(
            "They can be recreated from the cloud-masked, band-stacked L2A images and the saved model."
        )
        tile_log.info(
            "---------------------------------------------------------------"
        )
        directories = [
            categorised_image_dir,
            sieved_image_dir,
            probability_image_dir,
        ]
        for directory in directories:
            paths = [f for f in os.listdir(directory)]
            for f in paths:
                # keep the classified composite layers and the report image product for the next change detection
                if not f.startswith("composite_") and not f.startswith("report_"):
                    tile_log.info("Deleting {}".format(os.path.join(directory, f)))
                    if os.path.isdir(os.path.join(directory, f)):
                        shutil.rmtree(os.path.join(directory, f))
                    else:
                        os.remove(os.path.join(directory, f))
        tile_log.info(
            "---------------------------------------------------------------"
        )
        tile_log.info("Deletion of intermediate file products complete.")
        tile_log.info(
            "---------------------------------------------------------------"
        )
    else:
        if config_dict["do_zip"]:
            tile_log.info(
                "---------------------------------------------------------------"
            )
            tile_log.info(
                "Zipping intermediate class images used in change detection"
            )
            tile_log.info(
                "---------------------------------------------------------------"
            )
            directories = [categorised_image_dir, sieved_image_dir]
            for directory in directories:
                filesystem_utilities.zip_contents(
                    directory, notstartswith=["composite_", "report_"]
                )
            tile_log.info(
                "---------------------------------------------------------------"
            )
            tile_log.info("Zipping complete")
            tile_log.info(
                "---------------------------------------------------------------"
            )

    tile_log.info("---------------------------------------------------------------")
    tile_log.info(
        "Change detection and report image product updating, file compression, zipping"
    )
    tile_log.info(
        "and deletion of intermediate file products (if selected) are complete."
    )
    tile_log.info("---------------------------------------------------------------")

    if config_dict["do_delete"]:
        tile_log.info("---------------------------------------------------------------")
        tile_log.info("Deleting temporary directories starting with 'tmp*'")
        tile_log.info("These can be left over from interrupted processing runs.")
        tile_log.info("---------------------------------------------------------------")
        directory = tile_root_dir
        for root, dirs, files in os.walk(directory):
            temp_dirs = [d for d in dirs if d.startswith("tmp")]
            for temp_dir in temp_dirs:
                tile_log.info("Deleting {}".format(os.path.join(root, temp_dir)))
                if os.path.isdir(os.path.join(directory, f)):
                    shutil.rmtree(os.path.join(directory, f))
                else:
                    tile_log.warning(
                        "This should not have happened. {} is not a directory. Skipping deletion.".format(
                            os.path.join(root, temp_dir)
                        )
                    )
        tile_log.info("---------------------------------------------------------------")
        tile_log.info("Deletion of temporary directories complete.")
        tile_log.info("---------------------------------------------------------------")


tile_log.info("---------------------------------------------------------------")
tile_log.info("---                  PROCESSING END                         ---")
tile_log.info("---------------------------------------------------------------")

2026-07-30 02:41:49,942: INFO: ---------------------------------------------------------------
2026-07-30 02:41:49,943: INFO: ---                  PROCESSING END                         ---
2026-07-30 02:41:49,945: INFO: ---------------------------------------------------------------
